# CAS Landslide segmentation — DPF-Net experiments (Kaggle, GPU T4 x2)

**DPF-Net (Dual-Prior Fusion Network)**: a ConvNeXt-T (DINOv3) UNet that combines

* a **spectral-structural prior**: 10 differentiable descriptors that feed a high-resolution stem;
* a **semantic prior**: multi-layer features of a **frozen satellite foundation model** (DINOv3 ViT-L/16, SAT-493M),
  injected into the 1/8, 1/16 and 1/32 encoder features by **structure-aware gated fusion**;
* **semantic hard-negative mining**: anchors of the pixel-contrastive loss chosen both by the current prediction
  errors and by foundation-model similarity to landslides (bare soil, river beds, roads, ...);
* **DPF-Lite**: the same foundation model used only during training (feature distillation + mining), so it adds no
  inference cost.

`--fm none` reproduces the earlier SP-UNet bit for bit.

**One notebook, all experiments.** Each *Save & Run All* trains the experiments in `QUEUE`, in order, until the 12 h
session is nearly used up. Every run checkpoints. To continue in a new session, attach this notebook's previous
output as an input and run again: finished runs are carried forward, an unfinished run resumes from its checkpoint,
and the tables and figures are rebuilt from everything found.

### Before running
1. *Settings → Accelerator → GPU T4 x2*, *Settings → Internet → On*.
2. *Add Input →* dataset **`ngxunninh/cas-landslide`**.
3. From the 2nd session on: *Add Input → Your Work →* this notebook (its latest output).
4. Optional, to reuse your earlier SP-UNet runs as rows A3 / U3 without retraining: *Add Input →* the notebook
   versions that produced them, then set `IMPORT` in CONFIG. They are re-evaluated (≈20 min each), not retrained.
5. Edit CONFIG → *Save Version → Save & Run All (Commit)*.

### Experiments
| ID | Subset | Experiment | Priority |
|---|---|---|---|
| `A0` | SAT | UNet + PPM baseline | essential |
| `A1` | SAT | + spectral-structural prior & high-res stem | recommended |
| `A2` | SAT | + boundary head | recommended |
| `A3` | SAT | + error-mined pixel contrast (= SP-UNet) | essential |
| `A4` | SAT | + foundation semantic prior, additive fusion | recommended |
| `A5` | SAT | + structure-aware gated fusion | essential |
| `A6` | SAT | + semantic hard-negative mining = DPF-Net | essential |
| `A7` | SAT | DPF-Net without structural prior / high-res stem | essential |
| `A8` | SAT | DPF-Lite: foundation model only during training | recommended |
| `A9` | SAT | DPF-Net with a web-image foundation model | recommended |
| `S1` | SAT | DPF-Net, seed 1 | optional |
| `S2` | SAT | DPF-Net, seed 2 | optional |
| `U0` | UAV | UAV: UNet + PPM baseline | recommended |
| `U3` | UAV | UAV: SP-UNet | essential |
| `U6` | UAV | UAV: DPF-Net | essential |
| `U8` | UAV | UAV: DPF-Lite | optional |

All rows share the encoder, the 80-epoch schedule, seed 42 and the evaluation protocol. The protocol: the
checkpoint, the decision threshold and the TTA mode are chosen on validation only; the ablation table compares D4-TTA
at the validation threshold.

**Cost.** Runs without a foundation model (A0–A3, U0, U3) take as long as your earlier SP-UNet runs. Runs with the
ViT-L foundation model take about 1.3–1.6× longer, because they add one frozen forward pass per image. UAV runs take
about 1.8× longer than SAT runs. After epoch 1 the log prints the measured epoch time and the projected total. A run
that does not fit in a session pauses and resumes in the next one.

Suggested order: `A6` (main SAT result) → `A0`, `A5`, `A7` → `U6` (main UAV result) → `A4`, `A8`, `A9` → `A1`,
`A2` → `S1`, `S2`, `U0`, `U8`. Import `A3` / `U3` from your SP-UNet runs. With two Kaggle accounts, give each a
different `QUEUE`, then run the collect cell on all outputs.

In [ ]:
import time
NB_T0 = time.time()          # session start - the queue keeps every run inside the 12 h Kaggle limit
# ================================ CONFIG ================================
QUEUE = ['A6']               # experiment ids, run in this order (table above); e.g. ['A6', 'A0', 'A5']
EPOCHS = 80                  # identical for every row (the SP-UNet results used 80)
BATCH_PER_GPU = 8            # global batch = 16 on T4 x2; use 6 if CUDA runs out of memory
SESSION_HOURS = 11.5         # Kaggle stops sessions at 12 h; the queue stops cleanly before this
EXTRA_ARGS = ''              # appended to every run, e.g. '--grad-ckpt'
IMPORT = {}                  # reuse finished SP-UNet checkpoints instead of retraining, e.g.
# IMPORT = {'A3': '/kaggle/input/<sat-notebook>/runs/sat_spunet_convnext_t_dinov3_fp32_rerun/best.pth',
#           'U3': '/kaggle/input/<uav-notebook>/runs/sat_spunet_convnext_t_dinov3_fp32_rerun/best.pth'}
QUICK_TEST = False           # True: 1 epoch of QUEUE[0] (+ its evaluation) to measure time and memory
WORK = '/kaggle/working'
PREV = ['/kaggle/input']     # earlier outputs attached as inputs are found here automatically

In [ ]:
%%writefile /kaggle/working/prepare_data.py
#!/usr/bin/env python3
"""
prepare_data.py - build train / val / test split CSVs for the CAS Landslide Dataset.

  SAT : the public SC-Net split (5,204 train / 2,218 test tiles) from
        https://github.com/LiaoChengCRCC/SC-Net/tree/main/datasets
        -> identical test set to SC-Net (Song et al., Remote Sens. 2026, 18(13), 2216).
  UAV : seeded random 7:3 split per region (13,443 tiles in 7 UAV regions),
        same protocol as Xu et al. (Sci. Data 2024). Save the CSVs and publish them.

A region-stratified validation subset (default 10 %) is carved out of the training
split.  The test split is never used for model selection.

Outputs  <out>/<SUBSET>/train.csv     training tiles (train minus val)
         <out>/<SUBSET>/val.csv       validation tiles (model selection / threshold)
         <out>/<SUBSET>/trainval.csv  train + val (for --full-train final runs)
         <out>/<SUBSET>/test.csv      held-out test tiles
         <out>/<SUBSET>/summary.json  counts per region, split hash, source folders
CSV columns: name, region, img, mask   (img / mask are absolute paths)

Works with the Zenodo record 10294997 folder layout
    <root>/<Region name>/{img,label,mask}/<tile>.tif
(e.g. the Kaggle mirror "ngxunninh/cas-landslide").  Region folders are found
automatically anywhere below --input-root.
"""
import argparse
import csv
import hashlib
import io
import json
import math
import os
import random
import time
import unicodedata
import urllib.request
from collections import OrderedDict, defaultdict, deque
from pathlib import Path

import cv2
import numpy as np

IMG_EXT = {'.tif', '.tiff', '.png', '.jpg', '.jpeg', '.bmp'}

# normalised folder-name prefix -> readable region name
SAT_REGIONS = OrderedDict([
    ('tiburonpeninsulasentinel', 'Tiburon Peninsula (Sentinel)'),
    ('tiburonpeninsulaplanet', 'Tiburon Peninsula (Planet)'),
    ('hokkaidoiburitobu', 'Hokkaido Iburi-Tobu'),
    ('lombok', 'Lombok'),
    ('longxiriversat', 'Longxi River (SAT)'),
    ('mengdong', 'Mengdong'),
    ('moxitaidisat', 'Moxitaidi (SAT)'),
    ('palu', 'Palu'),
    ('wenchuan', 'Wenchuan'),
])
UAV_REGIONS = OrderedDict([
    ('jiuzhaivalleyuav02m', 'Jiuzhai Valley (UAV-0.2m)'),
    ('jiuzhaivalleyuav05m', 'Jiuzhai Valley (UAV-0.5m)'),
    ('longxiriveruav', 'Longxi River (UAV)'),
    ('moxitownuav02m', 'Moxi Town (UAV-0.2m)'),
    ('moxitownuav1m', 'Moxi Town (UAV-1m)'),
    ('moxitaidiuav06m', 'Moxitaidi (UAV-0.6m)'),
    ('moxitaidiuav1m', 'Moxitaidi (UAV-1m)'),
])
EXPECTED_SAT = {'train': 5204, 'test': 2218}
EXPECTED_UAV_TOTAL = 13443
SCNET_URLS = {
    'train': 'https://raw.githubusercontent.com/LiaoChengCRCC/SC-Net/HEAD/datasets/train.csv',
    'test': 'https://raw.githubusercontent.com/LiaoChengCRCC/SC-Net/HEAD/datasets/test.csv',
}
SKIP_DIRS = {'img', 'imgs', 'image', 'images', 'mask', 'masks', 'label', 'labels'}


def norm_name(s):
    """NFKC-normalise, case-fold and keep only alphanumerics (handles full-width parentheses)."""
    return ''.join(c for c in unicodedata.normalize('NFKC', s).casefold() if c.isalnum())


def find_region_dirs(input_root, regions, max_depth=8):
    """Breadth-first search for folders whose normalised name starts with a region key
    and that contain both an 'img' and a 'mask' sub-folder."""
    found = defaultdict(list)
    queue = deque([(Path(input_root), 0)])
    while queue:
        d, depth = queue.popleft()
        try:
            subdirs = [e for e in os.scandir(d) if e.is_dir()]
        except (PermissionError, FileNotFoundError, NotADirectoryError, OSError):
            continue
        lower = {e.name.lower() for e in subdirs}
        nd = norm_name(d.name)
        if 'img' in lower and 'mask' in lower:
            for key in regions:
                if nd.startswith(key):
                    found[key].append(d)
        if depth < max_depth:
            for e in subdirs:
                if e.name.lower() in SKIP_DIRS:
                    continue
                queue.append((Path(e.path), depth + 1))
    return found


def pick_dir(key, candidates):
    cands = sorted(candidates, key=lambda p: (len(p.parts), str(p)))
    if len(cands) > 1:
        print(f'  WARNING: {len(cands)} folders match region "{key}", using the first:')
        for c in cands:
            print('     ', c)
    return cands[0]


def child_dir(region_dir, name):
    for e in os.scandir(region_dir):
        if e.is_dir() and e.name.lower() == name:
            return Path(e.path)
    raise FileNotFoundError(f'No "{name}" folder in {region_dir}')


def index_images(folder):
    """stem -> absolute path for all image files below folder (hidden files ignored)."""
    out, dups = {}, []
    for dirpath, _, files in os.walk(folder):
        for f in files:
            if f.startswith('.'):
                continue
            stem, ext = os.path.splitext(f)
            if ext.lower() not in IMG_EXT:
                continue
            if stem in out:
                dups.append(stem)
            out[stem] = str(Path(dirpath, f).resolve())
    return out, dups


def index_region(region_dir):
    imgs, d1 = index_images(child_dir(region_dir, 'img'))
    masks, d2 = index_images(child_dir(region_dir, 'mask'))
    if d1 or d2:
        raise ValueError(f'Duplicate tile stems inside {region_dir}: {(d1 + d2)[:10]}')
    common = sorted(set(imgs) & set(masks))
    no_mask = sorted(set(imgs) - set(masks))
    no_img = sorted(set(masks) - set(imgs))
    return {s: (imgs[s], masks[s]) for s in common}, no_mask, no_img


def read_text(url_or_path, retries=3):
    if os.path.exists(url_or_path):
        return Path(url_or_path).read_text(encoding='utf-8-sig')
    last = None
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url_or_path, timeout=60) as r:
                return r.read().decode('utf-8-sig')
        except Exception as e:  # noqa: BLE001
            last = e
            time.sleep(2 + 3 * attempt)
    raise RuntimeError(f'Could not download {url_or_path}: {last}. '
                       'Turn Internet ON, or pass --csv-dir with SC-Net train.csv/test.csv.')


def scnet_stems(split, csv_dir):
    src = str(Path(csv_dir) / f'{split}.csv') if csv_dir else SCNET_URLS[split]
    text = read_text(src)
    rows = list(csv.DictReader(io.StringIO(text)))
    if not rows or 'img' not in rows[0]:
        raise ValueError(f'{src}: expected a CSV with an "img" column')
    stems = [Path(r['img'].strip().replace('\\', '/')).stem for r in rows]
    if len(set(stems)) != len(stems):
        raise ValueError(f'{src}: duplicate tile names')
    return stems, src, text


def round_half_up(x):
    return int(math.floor(x + 0.5))


def make_row(name, region, pair):
    return {'name': name, 'region': region, 'img': pair[0], 'mask': pair[1]}


def stratified_val(rows, frac, seed):
    """Carve a region-stratified validation subset out of the training rows."""
    by_region = defaultdict(list)
    for r in rows:
        by_region[r['region']].append(r)
    fit, val = [], []
    for region in sorted(by_region):
        rr = sorted(by_region[region], key=lambda r: r['name'])
        random.Random(f'{seed}-val-{region}').shuffle(rr)
        n_val = round_half_up(len(rr) * frac) if frac > 0 else 0
        n_val = min(n_val, max(len(rr) - 1, 0))
        val += rr[:n_val]
        fit += rr[n_val:]
    return fit, val


def write_csv(path, rows):
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['name', 'region', 'img', 'mask'])
        w.writeheader()
        for r in rows:
            w.writerow(r)


def split_hash(splits):
    h = hashlib.sha256()
    for name in ('train', 'val', 'test'):
        for r in sorted(splits[name], key=lambda r: r['name']):
            h.update(f'{name}:{r["region"]}:{r["name"]}\n'.encode('utf-8'))
    return h.hexdigest()


def imread_any(path):
    buf = np.fromfile(path, dtype=np.uint8)
    if buf.size == 0:
        return None
    return cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)


def sanity_check(rows, n, seed):
    rng = random.Random(seed)
    sample = rng.sample(rows, min(n, len(rows)))
    shapes, dtypes, chans, mvals = defaultdict(int), defaultdict(int), defaultdict(int), defaultdict(int)
    fg, problems = [], []
    for r in sample:
        img, m = imread_any(r['img']), imread_any(r['mask'])
        if img is None or m is None:
            problems.append(f'unreadable: {r["img"]} / {r["mask"]}')
            continue
        shapes[str(img.shape[:2])] += 1
        dtypes[str(img.dtype)] += 1
        chans[1 if img.ndim == 2 else img.shape[2]] += 1
        mm = m[..., 0] if m.ndim == 3 else m
        for v in np.unique(mm)[:6].tolist():
            mvals[v] += 1
        if mm.shape != img.shape[:2]:
            problems.append(f'mask/image size mismatch: {r["name"]} {mm.shape} vs {img.shape[:2]}')
        thr = 0 if mm.max() <= 1 else 127
        fg.append(float((mm > thr).mean()))
    print(f'  sanity check on {len(sample)} random tiles:')
    print(f'    image sizes   : {dict(shapes)}')
    print(f'    image dtypes  : {dict(dtypes)}    channels: {dict(chans)}')
    print(f'    mask values   : {dict(mvals)}')
    if fg:
        print(f'    landslide px  : mean {100 * np.mean(fg):.2f} %  (min {100 * np.min(fg):.2f} %, max {100 * np.max(fg):.2f} %)')
    for p in problems[:10]:
        print('    PROBLEM:', p)
    bad_vals = [v for v in mvals if v not in (0, 1, 255)]
    if bad_vals:
        print(f'    NOTE: unexpected mask values {bad_vals[:6]}; masks are binarised as >0 (0/1) or >127 (0/255).')
    return problems


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument('--subset', choices=['SAT', 'UAV'], default='SAT')
    p.add_argument('--input-root', default='/kaggle/input', help='folder searched for the CAS region folders')
    p.add_argument('--out', default='/kaggle/working/splits')
    p.add_argument('--val-frac', type=float, default=0.1, help='fraction of each region\'s training tiles used for validation')
    p.add_argument('--test-frac', type=float, default=0.3, help='UAV only: test fraction per region')
    p.add_argument('--seed', type=int, default=2024, help='seed of the validation split (and of the UAV split)')
    p.add_argument('--csv-dir', default='', help='SAT only: local folder with SC-Net train.csv / test.csv (no download)')
    p.add_argument('--check-n', type=int, default=40, help='number of random tiles to sanity-check')
    p.add_argument('--allow-count-mismatch', action='store_true')
    a = p.parse_args(argv)

    t0 = time.time()
    regions = SAT_REGIONS if a.subset == 'SAT' else UAV_REGIONS
    print(f'[prepare] subset={a.subset}  searching region folders below {a.input_root} ...', flush=True)
    found = find_region_dirs(a.input_root, regions)
    missing = [k for k in regions if not found.get(k)]
    if missing:
        raise SystemExit(
            f'Could not find the {a.subset} region folders {missing} below {a.input_root}.\n'
            'Attach the CAS Landslide dataset (e.g. Kaggle "ngxunninh/cas-landslide" or the Zenodo 10294997 zips '
            'extracted) so that each region folder contains img/ and mask/ sub-folders.')

    pairs, region_dir = {}, {}
    for key, rname in regions.items():
        d = pick_dir(key, found[key])
        region_dir[rname] = str(d)
        pr, no_mask, no_img = index_region(d)
        pairs[rname] = pr
        msg = f'  {rname:30s} {len(pr):6d} tiles   <- {d}'
        if no_mask or no_img:
            msg += f'   (ignored: {len(no_mask)} images without mask, {len(no_img)} masks without image)'
        print(msg, flush=True)

    if a.subset == 'SAT':
        stem_map, clashes = {}, set()
        for rname, pr in pairs.items():
            for stem, pair in pr.items():
                if stem in stem_map:
                    clashes.add(stem)
                stem_map[stem] = (rname, pair)
        tr_stems, tr_src, tr_text = scnet_stems('train', a.csv_dir)
        te_stems, te_src, te_text = scnet_stems('test', a.csv_dir)
        overlap = set(tr_stems) & set(te_stems)
        if overlap:
            raise SystemExit(f'SC-Net train/test overlap: {sorted(overlap)[:10]}')
        for split, stems in (('train', tr_stems), ('test', te_stems)):
            if len(stems) != EXPECTED_SAT[split]:
                msg = f'SC-Net {split}.csv has {len(stems)} tiles, expected {EXPECTED_SAT[split]}'
                if not a.allow_count_mismatch:
                    raise SystemExit(msg + ' (use --allow-count-mismatch to continue)')
                print('  WARNING:', msg)
        need = set(tr_stems) | set(te_stems)
        amb = sorted(need & clashes)
        if amb:
            raise SystemExit(f'Ambiguous tile names found in several SAT regions: {amb[:10]}')
        lost = sorted(s for s in need if s not in stem_map)
        if lost:
            raise SystemExit(f'{len(lost)} SC-Net tiles not found in the dataset, e.g. {lost[:10]}')
        train_rows = [make_row(s, *stem_map[s]) for s in tr_stems]
        test_rows = [make_row(s, *stem_map[s]) for s in te_stems]
        source = {'train_csv': tr_src, 'test_csv': te_src,
                  'train_csv_sha256': hashlib.sha256(tr_text.encode('utf-8')).hexdigest(),
                  'test_csv_sha256': hashlib.sha256(te_text.encode('utf-8')).hexdigest()}
    else:
        train_rows, test_rows = [], []
        for key, rname in regions.items():
            stems = sorted(pairs[rname])
            random.Random(f'{a.seed}-uav-{key}').shuffle(stems)
            n_test = round_half_up(len(stems) * a.test_frac)
            for i, s in enumerate(stems):
                row = make_row(f'{key}__{s}', rname, pairs[rname][s])
                (test_rows if i < n_test else train_rows).append(row)
        total = len(train_rows) + len(test_rows)
        if total != EXPECTED_UAV_TOTAL:
            msg = f'UAV subset has {total} tiles, expected {EXPECTED_UAV_TOTAL}'
            if not a.allow_count_mismatch:
                raise SystemExit(msg + ' (use --allow-count-mismatch to continue)')
            print('  WARNING:', msg)
        source = {'protocol': f'random {1 - a.test_frac:.0%}/{a.test_frac:.0%} split per region, seed {a.seed}'}

    fit_rows, val_rows = stratified_val(train_rows, a.val_frac, a.seed)
    splits = {'train': fit_rows, 'val': val_rows, 'trainval': fit_rows + val_rows, 'test': test_rows}
    names = [r['name'] for r in fit_rows + val_rows + test_rows]
    assert len(names) == len(set(names)), 'duplicate tile names across splits'

    out = Path(a.out) / a.subset
    out.mkdir(parents=True, exist_ok=True)
    for name, rows in splits.items():
        write_csv(out / f'{name}.csv', rows)

    counts = {name: dict(sorted(((rg, sum(r['region'] == rg for r in rows)) for rg in pairs), key=lambda kv: kv[0]))
              for name, rows in splits.items()}
    summary = {
        'subset': a.subset, 'seed': a.seed, 'val_frac': a.val_frac,
        'n': {k: len(v) for k, v in splits.items()},
        'per_region': counts, 'region_dirs': region_dir, 'split_hash': split_hash(splits),
        'source': source, 'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    (out / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

    print(f'\n[prepare] {a.subset} split written to {out}')
    print(f'  {"region":30s} {"train":>7s} {"val":>6s} {"test":>6s}')
    for rg in sorted(pairs):
        print(f'  {rg:30s} {counts["train"][rg]:7d} {counts["val"][rg]:6d} {counts["test"][rg]:6d}')
    print(f'  {"TOTAL":30s} {len(fit_rows):7d} {len(val_rows):6d} {len(test_rows):6d}')
    print(f'  split hash: {summary["split_hash"][:16]}...')
    problems = sanity_check(fit_rows + val_rows + test_rows, a.check_n, a.seed)
    if problems:
        print('  Sanity check reported problems (see above). Training will still try to read every tile.')
    print(f'[prepare] done in {time.time() - t0:.1f} s', flush=True)
    return summary


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/train_landslide.py
#!/usr/bin/env python3
"""
train_landslide.py - DPF-Net (Dual-Prior Fusion Network) for CAS Landslide segmentation (SAT / UAV).

Model  : timm CNN encoder (default ConvNeXt-T, DINOv3) + PPM context + scSE UNet decoder, with two priors:
         * structural prior - 10 fixed spectral-structural descriptors (FP32) -> high-resolution stem;
         * semantic prior   - multi-layer features of a frozen remote-sensing foundation ViT
           (default DINOv3 ViT-L/16 SAT-493M) injected into the 1/8, 1/16, 1/32 encoder features by
           structure-aware gated fusion ('fuse'), or used only during training ('distill', no inference cost);
         + boundary head + deep supervision + pixel contrastive head with error- and foundation-guided
           hard-anchor mining.  '--fm none' gives the earlier SP-UNet exactly.
Train  : DDP (torchrun), FP16 autocast + dynamic GradScaler, AdamW, warmup + cosine,
         EMA weights, GPU augmentation, time budget with resume.
Eval   : EMA weights, dataset-level confusion counts, D4 (+ optional multi-scale) test-time augmentation,
         TTA mode and threshold selected on the validation split only, per-region and per-tile counts.

Kaggle (GPU T4 x2):
  NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 python -m torch.distributed.run --standalone \
      --nproc_per_node=2 train_landslide.py --splits /kaggle/working/splits/SAT \
      --out /kaggle/working/runs/sat_full
Single GPU / CPU:  python train_landslide.py --splits ... --out ...
"""
import argparse
import copy
import csv
import datetime
import hashlib
import json
import math
import os
import random
import re
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Sampler
from torch.utils.data.distributed import DistributedSampler

try:
    import timm
except ImportError as exc:  # pragma: no cover
    raise SystemExit('timm is required:  pip install timm') from exc

cv2.setNumThreads(0)
try:
    cv2.ocl.setUseOpenCL(False)
except Exception:  # noqa: BLE001
    pass

SWEEP_THRESHOLDS = [round(0.30 + 0.025 * i, 3) for i in range(17)]  # 0.300 ... 0.700 (contains 0.5)
ARCH_KEYS = ('encoder', 'img_size', 'no_hires', 'no_prior', 'no_attention', 'fm', 'fm_mode', 'fm_fusion',
             'fm_channels', 'fm_layers', 'fm_size')
# values assumed for arguments that checkpoints / results of the earlier SP-UNet version do not contain
LEGACY_DEFAULTS = {'fm': 'none', 'fm_mode': 'fuse', 'fm_fusion': 'gated', 'fm_channels': 256, 'fm_layers': 4,
                   'fm_size': 0, 'hard_mining': 'error', 'distill_weight': 0.0, 'ms_tta': False,
                   'ms_scales': [0.75, 1.0, 1.25], 'tag': ''}
TRAIN_ONLY_HEADS = ('aux4', 'aux8', 'bnd_head', 'proj', 'distill_head')
MODE_LABELS = {'noTTA': 'no TTA', 'TTA': 'D4-TTA', 'MSTTA': 'D4+MS-TTA'}


def arg(ns, key):
    """Argument value of a Namespace or dict, falling back to LEGACY_DEFAULTS for older checkpoints."""
    if isinstance(ns, dict):
        return ns.get(key, LEGACY_DEFAULTS.get(key))
    return getattr(ns, key, LEGACY_DEFAULTS.get(key))


def fm_enabled(name):
    return name not in (None, '', 'none')


# ----------------------------------------------------------------------------------------
# distributed helpers
# ----------------------------------------------------------------------------------------
class Dist:
    def __init__(self):
        self.world = int(os.environ.get('WORLD_SIZE', '1'))
        self.rank = int(os.environ.get('RANK', '0'))
        self.local_rank = int(os.environ.get('LOCAL_RANK', '0'))
        self.enabled = self.world > 1
        if torch.cuda.is_available():
            torch.cuda.set_device(self.local_rank)
            self.device = torch.device('cuda', self.local_rank)
        else:
            self.device = torch.device('cpu')
        if self.enabled:
            backend = 'nccl' if self.device.type == 'cuda' else 'gloo'
            dist.init_process_group(backend=backend, timeout=datetime.timedelta(minutes=60))

    @property
    def main(self):
        return self.rank == 0

    def barrier(self):
        if self.enabled:
            if self.device.type == 'cuda':
                dist.barrier(device_ids=[self.local_rank])
            else:
                dist.barrier()

    def sum_(self, t):
        if self.enabled:
            dist.all_reduce(t, op=dist.ReduceOp.SUM)
        return t

    def any_flag(self, flag):
        t = torch.tensor([1.0 if flag else 0.0], device=self.device)
        if self.enabled:
            dist.all_reduce(t, op=dist.ReduceOp.MAX)
        return bool(t.item() > 0.5)

    def close(self):
        if self.enabled and dist.is_initialized():
            dist.destroy_process_group()


def log(d, *args):
    if d.main:
        print(*args, flush=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed % (2 ** 32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save_atomic(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + '.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, path)


def write_json(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2), encoding='utf-8')
    os.replace(tmp, path)


def safe_name(s):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', s)


# ----------------------------------------------------------------------------------------
# data
# ----------------------------------------------------------------------------------------
def read_split(path):
    with open(path, newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        if not {'name', 'region', 'img', 'mask'} <= set(r):
            raise ValueError(f'{path}: expected columns name,region,img,mask')
    return rows


def rows_hash(*splits):
    h = hashlib.sha256()
    for i, rows in enumerate(splits):
        for r in sorted(rows, key=lambda r: r['name']):
            h.update(f'{i}:{r["region"]}:{r["name"]}\n'.encode('utf-8'))
    return h.hexdigest()


def imread_any(path):
    buf = np.fromfile(path, dtype=np.uint8)
    if buf.size == 0:
        raise IOError(f'Empty file: {path}')
    img = cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise IOError(f'Could not decode: {path}')
    return img


def to_uint8_rgb(img):
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    elif img.shape[2] == 1:
        img = np.repeat(img, 3, axis=2)
    elif img.shape[2] == 2:
        img = np.stack([img[:, :, 0]] * 3, axis=-1)
    elif img.shape[2] >= 4:
        img = img[:, :, :3]
    if img.dtype != np.uint8:
        x = img.astype(np.float32)
        lo, hi = np.percentile(x, 0.5), np.percentile(x, 99.5)
        x = (x - lo) / max(float(hi - lo), 1e-6)
        img = np.clip(x * 255.0 + 0.5, 0, 255).astype(np.uint8)
    return np.ascontiguousarray(img[:, :, ::-1])  # OpenCV BGR -> RGB


def to_binary_mask(m):
    if m.ndim == 3:
        m = m[:, :, 0]
    thr = 0 if m.max() <= 1 else 127
    return (m > thr).astype(np.uint8)


class SegDataset(Dataset):
    """Returns (uint8 3xHxW RGB, uint8 1xHxW mask, index). D4 flips/rotations for training;
    all other augmentation runs on the GPU (gpu_augment)."""

    def __init__(self, rows, img_size, train):
        self.rows, self.size, self.train = rows, img_size, train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        img = to_uint8_rgb(imread_any(r['img']))
        m = to_binary_mask(imread_any(r['mask']))
        if m.shape[:2] != img.shape[:2]:
            m = cv2.resize(m, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        s = self.size
        if img.shape[0] != s or img.shape[1] != s:
            interp = cv2.INTER_AREA if img.shape[0] > s else cv2.INTER_LINEAR
            img = cv2.resize(img, (s, s), interpolation=interp)
            m = cv2.resize(m, (s, s), interpolation=cv2.INTER_NEAREST)
        if self.train:
            k = random.randint(0, 3)
            if k:
                img, m = np.rot90(img, k), np.rot90(m, k)
            if random.random() < 0.5:
                img, m = img[:, ::-1], m[:, ::-1]
        img = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1).contiguous()
        m = torch.from_numpy(np.ascontiguousarray(m))[None]
        return img, m, i


class ShardSampler(Sampler):
    """Exact, non-overlapping sharding for distributed evaluation (no padded duplicates)."""

    def __init__(self, n, rank, world):
        self.idx = list(range(rank, n, world))

    def __iter__(self):
        return iter(self.idx)

    def __len__(self):
        return len(self.idx)


@torch.no_grad()
def gpu_augment(x, y, a):
    """x: Bx3xHxW float in [0,1]; y: Bx1xHxW float {0,1}. Geometric + photometric, per sample."""
    B, _, H, W = x.shape
    dev = x.device
    if a.aug_affine_p > 0:
        on = (torch.rand(B, device=dev) < a.aug_affine_p).float()
        ang = (torch.rand(B, device=dev) * 2 - 1) * math.radians(a.aug_rotate) * on
        lo, hi = math.log(a.aug_scale_min), math.log(a.aug_scale_max)
        sc = torch.exp(torch.empty(B, device=dev).uniform_(lo, hi) * on)
        tx = (torch.rand(B, device=dev) * 2 - 1) * a.aug_translate * on
        ty = (torch.rand(B, device=dev) * 2 - 1) * a.aug_translate * on
        cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
        theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
        grid = F.affine_grid(theta, [B, 3, H, W], align_corners=False)
        x = F.grid_sample(x, grid, mode='bilinear', padding_mode='reflection', align_corners=False)
        y = F.grid_sample(y, grid, mode='nearest', padding_mode='reflection', align_corners=False)
    if a.aug_color_p > 0:
        on = (torch.rand(B, 1, 1, 1, device=dev) < a.aug_color_p).float()

        def jitter(mag, shape=(B, 1, 1, 1)):
            return 1.0 + (torch.rand(shape, device=dev) * 2 - 1) * mag * on

        x = x * jitter(a.aug_brightness)
        mean = x.mean(dim=(1, 2, 3), keepdim=True)
        x = (x - mean) * jitter(a.aug_contrast) + mean
        gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
        x = (x - gray) * jitter(a.aug_saturation) + gray
        x = x * jitter(a.aug_channel_gain, (B, 3, 1, 1))
        x = x.clamp(0.0, 1.0)
    if a.aug_gamma_p > 0:
        on = (torch.rand(B, 1, 1, 1, device=dev) < a.aug_gamma_p).float()
        g = torch.exp(torch.empty(B, 1, 1, 1, device=dev).uniform_(math.log(0.8), math.log(1.25)) * on)
        x = x.clamp(min=1e-6).pow(g)
    if a.aug_noise_p > 0:
        on = (torch.rand(B, 1, 1, 1, device=dev) < a.aug_noise_p).float()
        sigma = torch.rand(B, 1, 1, 1, device=dev) * a.aug_noise_std * on
        x = (x + torch.randn_like(x) * sigma).clamp(0.0, 1.0)
    return x.contiguous(), y.contiguous()


# ----------------------------------------------------------------------------------------
# model
# ----------------------------------------------------------------------------------------
def _gauss1d(sigma):
    r = max(1, int(math.ceil(3.0 * sigma)))
    t = torch.arange(-r, r + 1, dtype=torch.float32)
    k = torch.exp(-0.5 * (t / sigma) ** 2)
    return k / k.sum()


class SpectralStructuralPrior(nn.Module):
    """Ten fixed, differentiable descriptors computed on the GPU in FP32 from RGB in [0,1]:
    gray, Sobel magnitude, LoG(s=1), DoG(1,2), DoG(2,4), local std (7x7, 15x15),
    local contrast normalisation, ExG vegetation index, saturation."""
    names = ('gray', 'sobel_mag', 'log_s1', 'dog_1_2', 'dog_2_4', 'local_std_7',
             'local_std_15', 'local_contrast', 'exg', 'saturation')
    out_channels = 10

    def __init__(self):
        super().__init__()
        sx = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]) / 8.0
        self.register_buffer('k_sobel', torch.stack([sx, sx.t()])[:, None].contiguous(), persistent=False)
        lap = torch.tensor([[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]])
        self.register_buffer('k_lap', lap[None, None].contiguous(), persistent=False)
        for s in (1, 2, 4):
            self.register_buffer(f'g{s}', _gauss1d(float(s)), persistent=False)

    @staticmethod
    def _blur(x, k):
        r = (k.numel() - 1) // 2
        x = F.conv2d(F.pad(x, (r, r, 0, 0), mode='reflect'), k.view(1, 1, 1, -1))
        return F.conv2d(F.pad(x, (0, 0, r, r), mode='reflect'), k.view(1, 1, -1, 1))

    @staticmethod
    def _box(x, w):
        r = w // 2
        return F.avg_pool2d(F.pad(x, (r, r, r, r), mode='reflect'), w, stride=1)

    def forward(self, rgb):
        rgb = rgb.float().contiguous()
        R, G, B = rgb[:, 0:1], rgb[:, 1:2], rgb[:, 2:3]
        gray = 0.299 * R + 0.587 * G + 0.114 * B
        gxy = F.conv2d(F.pad(gray, (1, 1, 1, 1), mode='reflect'), self.k_sobel)
        sobel = torch.sqrt(gxy.pow(2).sum(1, keepdim=True) + 1e-12)
        b1, b2, b4 = self._blur(gray, self.g1), self._blur(gray, self.g2), self._blur(gray, self.g4)
        log1 = F.conv2d(F.pad(b1, (1, 1, 1, 1), mode='reflect'), self.k_lap)

        def local_std(w):
            m = self._box(gray, w)
            m2 = self._box(gray * gray, w)
            return torch.sqrt((m2 - m * m).clamp(min=0.0) + 1e-8)

        std7, std15 = local_std(7), local_std(15)
        lcn = ((gray - b4) / (std15 + 1e-2)).clamp(-6.0, 6.0)
        exg = (2.0 * G - R - B) / (R + G + B + 1e-6)
        mx = rgb.amax(dim=1, keepdim=True)
        mn = rgb.amin(dim=1, keepdim=True)
        sat = (mx - mn) / (mx + 1e-6)
        return torch.cat([gray, sobel, log1, b1 - b2, b2 - b4, std7, std15, lcn, exg, sat], 1)


class ConvBNReLU(nn.Sequential):
    def __init__(self, cin, cout, k=3, stride=1):
        super().__init__(nn.Conv2d(cin, cout, k, stride, k // 2, bias=False),
                         nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class SCSE(nn.Module):
    """Concurrent spatial and channel squeeze & excitation."""

    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 8)
        self.cse = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(ch, mid, 1), nn.ReLU(inplace=True),
                                 nn.Conv2d(mid, ch, 1), nn.Sigmoid())
        self.sse = nn.Sequential(nn.Conv2d(ch, 1, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.cse(x) + x * self.sse(x)


class DecoderBlock(nn.Module):
    def __init__(self, cin, cskip, cout, attention=True):
        super().__init__()
        self.conv1 = ConvBNReLU(cin + cskip, cout)
        self.conv2 = ConvBNReLU(cout, cout)
        self.att = SCSE(cout) if attention else nn.Identity()

    def forward(self, x, skip=None):
        if skip is not None:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], 1)
        else:
            x = F.interpolate(x, scale_factor=2.0, mode='bilinear', align_corners=False)
        return self.att(self.conv2(self.conv1(x)))


class PPM(nn.Module):
    """Pyramid pooling context module (no BatchNorm on the pooled branches)."""

    def __init__(self, cin, cout, bins=(1, 2, 3, 6)):
        super().__init__()
        red = max(cin // len(bins), 64)
        self.stages = nn.ModuleList([nn.Sequential(nn.AdaptiveAvgPool2d(b), nn.Conv2d(cin, red, 1),
                                                   nn.ReLU(inplace=True)) for b in bins])
        self.bottleneck = ConvBNReLU(cin + red * len(bins), cout)

    def forward(self, x):
        h, w = x.shape[-2:]
        feats = [x] + [F.interpolate(s(x), size=(h, w), mode='bilinear', align_corners=False)
                       for s in self.stages]
        return self.bottleneck(torch.cat(feats, 1))


class Encoder(nn.Module):
    """timm features_only encoder returning the stride 4/8/16/32 feature maps."""

    def __init__(self, name, pretrained, grad_ckpt=False):
        super().__init__()
        self.body = timm.create_model(name, pretrained=pretrained, features_only=True)
        reds = list(self.body.feature_info.reduction())
        chs = list(self.body.feature_info.channels())
        self.idx = []
        for r in (4, 8, 16, 32):
            cands = [i for i, rr in enumerate(reds) if rr == r]
            if not cands:
                raise ValueError(f'Encoder {name} has no stride-{r} feature map (reductions {reds}).')
            self.idx.append(cands[-1])
        self.channels = [chs[i] for i in self.idx]
        cfg = getattr(self.body, 'pretrained_cfg', None) or {}
        mean = tuple(cfg.get('mean', (0.485, 0.456, 0.406)))
        std = tuple(cfg.get('std', (0.229, 0.224, 0.225)))
        if len(mean) != 3 or len(std) != 3:
            mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
        self.register_buffer('mean', torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1), persistent=False)
        self.register_buffer('std', torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1), persistent=False)
        if grad_ckpt and hasattr(self.body, 'set_grad_checkpointing'):
            self.body.set_grad_checkpointing(True)
        # sanity check: NCHW outputs with the advertised channels
        was_training = self.body.training
        self.body.eval()
        with torch.no_grad():
            feats = self.body(torch.zeros(1, 3, 64, 64))
        self.body.train(was_training)
        for i, c in zip(self.idx, self.channels):
            if feats[i].ndim != 4 or feats[i].shape[1] != c:
                raise ValueError(f'Encoder {name}: feature {i} has shape {tuple(feats[i].shape)}, expected NCHW '
                                 f'with {c} channels. Use a CNN encoder (ConvNeXt / RegNet / ResNet / EfficientNet).')

    def forward(self, x):
        feats = self.body(x)
        return [feats[i] for i in self.idx]


# ----------------------------------------------------------------------------------------
# foundation-model semantic prior
# ----------------------------------------------------------------------------------------
_FM_CACHE = {}


def fm_indices(depth, n):
    """n evenly spaced block indices ending at the last block (depth 24, n 4 -> [5, 11, 17, 23])."""
    n = max(1, min(int(n), depth))
    return sorted({max(0, int(round(depth * (i + 1) / n)) - 1) for i in range(n)})


def get_foundation(name, pretrained):
    """Frozen timm ViT shared by every model instance of this process. It lives outside the trainable module
    tree, so it is never trained, never copied into the EMA model, never wrapped by DDP and never written to
    checkpoints; loading a checkpoint re-creates it from timm. Creation does not consume the global RNG, so
    runs with and without a foundation model share the same random streams."""
    key = (name, bool(pretrained))
    vit = _FM_CACHE.get(key)
    if vit is None:
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(0)  # random weights (tests) are identical on every rank and every rebuild
            try:
                vit = timm.create_model(name, pretrained=pretrained, dynamic_img_size=True)
            except TypeError:
                vit = timm.create_model(name, pretrained=pretrained)
        if not hasattr(vit, 'forward_intermediates') or not hasattr(vit, 'blocks'):
            raise ValueError(f'--fm {name}: expected a timm ViT with forward_intermediates() '
                             '(e.g. vit_large_patch16_dinov3.sat493m).')
        vit.eval().requires_grad_(False)
        _FM_CACHE[key] = vit
    return vit


def _resize(f, size):
    """Resizes a feature map: average pooling when shrinking, bilinear interpolation otherwise."""
    size = (int(size[0]), int(size[1]))
    if tuple(f.shape[-2:]) == size:
        return f
    if f.shape[-2] >= size[0] and f.shape[-1] >= size[1]:
        return F.adaptive_avg_pool2d(f, size)
    return F.interpolate(f, size=size, mode='bilinear', align_corners=False)


class SemanticInjection(nn.Module):
    """Injects the foundation-model semantic map f into the CNN feature map c of one pyramid level.
      gated (structure-aware):  c' = c + g * P(f),  g = sigmoid(MLP([c, P(f), S])), where S is the
                                structural-prior stem (1/4) pooled to this level, when the stem exists;
      add:                      c' = c + P(f).
    P ends with a zero-initialised BatchNorm, so training starts exactly from the CNN-only network."""

    def __init__(self, c, cf, cs=0, gated=True):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(cf, c, 1, bias=False), nn.BatchNorm2d(c))
        nn.init.zeros_(self.proj[1].weight)
        self.cs = cs
        self.gate = None
        if gated:
            mid = max(c // 4, 16)
            self.gate = nn.Sequential(nn.Conv2d(2 * c + cs, mid, 1), nn.ReLU(inplace=True), nn.Conv2d(mid, c, 1))

    def forward(self, c, f, s=None):
        fp = self.proj(_resize(f, c.shape[-2:])).to(c.dtype)
        if self.gate is None:
            return c + fp
        parts = [c, fp]
        if self.cs:
            parts.append(_resize(s, c.shape[-2:]).to(c.dtype))
        g = torch.sigmoid(self.gate(torch.cat(parts, 1))).to(c.dtype)
        return c + g * fp


class DualPriorUNet(nn.Module):
    """UNet decoder over a timm CNN encoder with two optional priors (see the module docstring).
    fm=None reproduces the earlier Structural-Prior UNet (SP-UNet) exactly."""

    def __init__(self, encoder='convnext_tiny.dinov3_lvd1689m', pretrained=True, hires=True, prior=True,
                 boundary=True, contrast=True, deep_sup=True, attention=True, proj_dim=64, grad_ckpt=False,
                 fm=None, fm_pretrained=True, fm_mode='fuse', fm_fusion='gated', fm_channels=256, fm_layers=4,
                 fm_scale=1.0):
        super().__init__()
        if prior and not hires:
            raise ValueError('The structural prior needs the high-resolution stem (use --no-prior with --no-hires).')
        if fm_mode not in ('fuse', 'distill') or fm_fusion not in ('gated', 'add'):
            raise ValueError(f'unknown fm_mode / fm_fusion: {fm_mode} / {fm_fusion}')
        self.hires, self.use_prior = hires, prior
        self.encoder = Encoder(encoder, pretrained, grad_ckpt)
        c4, c8, c16, c32 = self.encoder.channels
        s1 = s2 = s4 = 0
        if hires:
            cin = 3
            if prior:
                self.prior = SpectralStructuralPrior()
                self.prior_bn = nn.BatchNorm2d(SpectralStructuralPrior.out_channels)
                cin += SpectralStructuralPrior.out_channels
            self.stem1 = nn.Sequential(ConvBNReLU(cin, 16), ConvBNReLU(16, 16))
            self.stem2 = nn.Sequential(ConvBNReLU(16, 32, stride=2), ConvBNReLU(32, 32))
            self.stem4 = nn.Sequential(ConvBNReLU(32, 48, stride=2), ConvBNReLU(48, 48))
            s1, s2, s4 = 16, 32, 48
        self.ppm = PPM(c32, 256)
        self.dec16 = DecoderBlock(256, c16, 256, attention)
        self.dec8 = DecoderBlock(256, c8, 128, attention)
        self.dec4 = DecoderBlock(128, c4 + s4, 64, attention)
        self.dec2 = DecoderBlock(64, s2, 32, attention)
        self.dec1 = DecoderBlock(32, s1, 16, attention)
        self.seg_head = nn.Sequential(ConvBNReLU(16, 16), nn.Conv2d(16, 1, 1))
        self.aux4 = nn.Conv2d(64, 1, 1) if deep_sup else None
        self.aux8 = nn.Conv2d(128, 1, 1) if deep_sup else None
        self.bnd_head = nn.Sequential(ConvBNReLU(16, 16), nn.Conv2d(16, 1, 1)) if boundary else None
        self.proj = (nn.Sequential(nn.Conv2d(64, 64, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
                                   nn.Conv2d(64, proj_dim, 1)) if contrast else None)
        nn.init.constant_(self.seg_head[-1].bias, -2.0)
        if deep_sup:
            nn.init.constant_(self.aux4.bias, -2.0)
            nn.init.constant_(self.aux8.bias, -2.0)
        if boundary:
            nn.init.constant_(self.bnd_head[-1].bias, -3.0)
        # ---- semantic prior (created after all SP-UNet modules, so their initialisation is unchanged) ----
        self.fm_name = fm if fm_enabled(fm) else None
        self.fm_mode, self.fm_pretrained, self.fm_scale = fm_mode, bool(fm_pretrained), float(fm_scale)
        self.fm_fallbacks = 0
        self.distill_head = None
        if self.fm_name:
            vit = get_foundation(self.fm_name, self.fm_pretrained)
            ps = vit.patch_embed.patch_size
            self.fm_patch = int(ps[0] if isinstance(ps, (tuple, list)) else ps)
            self.fm_idx = fm_indices(len(vit.blocks), fm_layers)
            self.fm_dim = int(vit.embed_dim)
            cfg = getattr(vit, 'pretrained_cfg', None) or {}
            mean = tuple(cfg.get('mean', (0.485, 0.456, 0.406)))
            std = tuple(cfg.get('std', (0.229, 0.224, 0.225)))
            if len(mean) != 3 or len(std) != 3:
                mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
            self.register_buffer('fm_mean', torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1), persistent=False)
            self.register_buffer('fm_std', torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1), persistent=False)
            if fm_mode == 'fuse':
                gated = fm_fusion == 'gated'
                self.fm_adapter = nn.Sequential(
                    nn.Conv2d(self.fm_dim * len(self.fm_idx), fm_channels, 1, bias=False),
                    nn.BatchNorm2d(fm_channels), nn.GELU(),
                    nn.Conv2d(fm_channels, fm_channels, 3, padding=1, bias=False),
                    nn.BatchNorm2d(fm_channels), nn.GELU())
                cs = s4 if gated else 0
                self.inj8 = SemanticInjection(c8, fm_channels, cs, gated)
                self.inj16 = SemanticInjection(c16, fm_channels, cs, gated)
                self.inj32 = SemanticInjection(c32, fm_channels, cs, gated)
            else:  # distill: predict the foundation features from the CNN's 1/16 features (training only)
                self.distill_head = nn.Sequential(nn.Conv2d(c16, 256, 1, bias=False), nn.BatchNorm2d(256), nn.GELU(),
                                                  nn.Conv2d(256, self.fm_dim, 1))

    @torch.no_grad()
    def foundation_features(self, x):
        """Frozen-ViT features of the blocks in self.fm_idx (final LayerNorm applied), FP32 NCHW at 1/patch of
        the foundation-model input (x resized by fm_scale, multiple of the patch size, own normalisation)."""
        vit = get_foundation(self.fm_name, self.fm_pretrained)
        if next(vit.parameters()).device != x.device:
            vit.to(x.device)
        H, W = x.shape[-2:]
        ps = self.fm_patch
        h = max(ps, int(round(H * self.fm_scale / ps)) * ps)
        w = max(ps, int(round(W * self.fm_scale / ps)) * ps)
        xi = x.float()
        if (h, w) != (H, W):
            xi = F.interpolate(xi, size=(h, w), mode='bilinear', align_corners=False, antialias=h < H)
        xi = (xi - self.fm_mean) / self.fm_std
        kw = dict(indices=self.fm_idx, norm=True, output_fmt='NCHW', intermediates_only=True)
        feats = [f.float() for f in vit.forward_intermediates(xi, **kw)]
        if not bool(torch.stack([torch.isfinite(f).all() for f in feats]).all()):
            with torch.autocast(device_type=x.device.type, enabled=False):  # FP16 overflow: recompute in FP32
                feats = [f.float() for f in vit.forward_intermediates(xi, **kw)]
            self.fm_fallbacks += 1
        return feats

    def forward(self, x):
        """x: Bx3xHxW RGB in [0,1] (float32). Training: dict with seg/aux4/aux8/bnd/emb (+ fm_sem / distill
        with a foundation model); eval: {'seg'}."""
        H, W = x.shape[-2:]
        xn = (x - self.encoder.mean) / self.encoder.std
        c4, c8, c16, c32 = self.encoder(xn)
        p1 = p2 = p4 = None
        if self.hires:
            s_in = xn
            if self.use_prior:
                with torch.autocast(device_type=x.device.type, enabled=False):
                    pr = self.prior_bn(self.prior(x.float()))
                s_in = torch.cat([xn.float(), pr], 1)
            p1 = self.stem1(s_in)
            p2 = self.stem2(p1)
            p4 = self.stem4(p2)
            if p4.shape[-2:] != c4.shape[-2:]:
                p4 = F.interpolate(p4, size=c4.shape[-2:], mode='bilinear', align_corners=False)
        fm_sem = None
        if self.fm_name and (self.fm_mode == 'fuse' or self.training):
            feats = self.foundation_features(x)
            if self.training:
                fm_sem = F.normalize(feats[-1], dim=1)
            if self.fm_mode == 'fuse':
                f = self.fm_adapter(torch.cat(feats, 1))
                c8 = self.inj8(c8, f, p4)
                c16 = self.inj16(c16, f, p4)
                c32 = self.inj32(c32, f, p4)
        d16 = self.dec16(self.ppm(c32), c16)
        d8 = self.dec8(d16, c8)
        d4 = self.dec4(d8, c4 if p4 is None else torch.cat([c4, p4.to(c4.dtype)], 1))
        d2 = self.dec2(d4, p2)
        d1 = self.dec1(d2, p1)
        seg = self.seg_head(d1)
        if seg.shape[-2:] != (H, W):
            seg = F.interpolate(seg, size=(H, W), mode='bilinear', align_corners=False)
        out = {'seg': seg}
        if self.training:
            if self.aux4 is not None:
                out['aux4'] = self.aux4(d4)
                out['aux8'] = self.aux8(d8)
            if self.bnd_head is not None:
                b = self.bnd_head(d1)
                if b.shape[-2:] != (H, W):
                    b = F.interpolate(b, size=(H, W), mode='bilinear', align_corners=False)
                out['bnd'] = b
            if self.proj is not None:
                out['emb'] = self.proj(d4)
            if fm_sem is not None:
                out['fm_sem'] = fm_sem
            if self.distill_head is not None:
                out['distill'] = self.distill_head(c16)
        return out


class Ensemble(nn.Module):
    """Averages member probabilities; returns logits of the mean probability."""

    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleList(models)

    def forward(self, x):
        p = torch.stack([torch.sigmoid(m(x)['seg'].float()) for m in self.models]).mean(0)
        return {'seg': torch.logit(p.clamp(1e-6, 1 - 1e-6))}


def build_model(a, pretrained):
    """a: argparse Namespace of this or an earlier version (missing keys -> LEGACY_DEFAULTS). The foundation
    model is always loaded from timm (pretrained unless the run used --no-pretrained); it is not in checkpoints."""
    fm_size = int(arg(a, 'fm_size') or 0)
    return DualPriorUNet(
        encoder=a.encoder, pretrained=pretrained, hires=not a.no_hires, prior=not a.no_prior,
        boundary=a.boundary_weight > 0, contrast=a.contrast_weight > 0,
        deep_sup=any(w > 0 for w in a.aux_weights), attention=not a.no_attention,
        grad_ckpt=getattr(a, 'grad_ckpt', False),
        fm=arg(a, 'fm'), fm_pretrained=not getattr(a, 'no_pretrained', False), fm_mode=arg(a, 'fm_mode'),
        fm_fusion=arg(a, 'fm_fusion'), fm_channels=int(arg(a, 'fm_channels')), fm_layers=int(arg(a, 'fm_layers')),
        fm_scale=(fm_size / a.img_size) if fm_size else 1.0)


# ----------------------------------------------------------------------------------------
# losses
# ----------------------------------------------------------------------------------------
def bce_dice(logits, target, smooth=1.0):
    logits = logits.float()
    bce = F.binary_cross_entropy_with_logits(logits, target)
    p = torch.sigmoid(logits)
    dice = 1.0 - (2.0 * (p * target).sum() + smooth) / (p.sum() + target.sum() + smooth)
    return bce + dice


def boundary_target(mask, k=3):
    dil = F.max_pool2d(mask, k, 1, k // 2)
    ero = -F.max_pool2d(-mask, k, 1, k // 2)
    return (dil - ero).clamp(0.0, 1.0)


def _sample_idx(cls_mask, hard_mask, n):
    hard = hard_mask.nonzero(as_tuple=False).squeeze(1)
    easy = (cls_mask & ~hard_mask).nonzero(as_tuple=False).squeeze(1)
    n_hard = min(hard.numel(), n // 2)
    n_easy = min(easy.numel(), n - n_hard)
    n_hard = min(hard.numel(), n - n_easy)
    sel = []
    if n_hard > 0:
        sel.append(hard[torch.randperm(hard.numel(), device=hard.device)[:n_hard]])
    if n_easy > 0:
        sel.append(easy[torch.randperm(easy.numel(), device=easy.device)[:n_easy]])
    return torch.cat(sel) if sel else hard[:0]


def _top_frac(cls_mask, score, frac):
    """Boolean mask of the top `frac` fraction (by score) of the pixels in cls_mask."""
    out = torch.zeros_like(cls_mask)
    idx = cls_mask.nonzero(as_tuple=False).squeeze(1)
    if idx.numel() == 0:
        return out
    k = max(1, int(math.ceil(frac * idx.numel())))
    out[idx[torch.topk(score[idx], k).indices]] = True
    return out


def _sample_groups(cls_mask, groups, n):
    """Up to n distinct pixels of one class: n // (len(groups) + 1) from each hard group in turn, the rest
    uniformly from the remaining pixels of the class."""
    avail = cls_mask.clone()
    quota = n // (len(groups) + 1)
    sel = []
    for g in groups:
        cand = (g & avail).nonzero(as_tuple=False).squeeze(1)
        k = min(cand.numel(), quota)
        if k > 0:
            pick = cand[torch.randperm(cand.numel(), device=cand.device)[:k]]
            avail[pick] = False
            sel.append(pick)
    rest = avail.nonzero(as_tuple=False).squeeze(1)
    k = min(rest.numel(), n - sum(int(s.numel()) for s in sel))
    if k > 0:
        sel.append(rest[torch.randperm(rest.numel(), device=rest.device)[:k]])
    return torch.cat(sel) if sel else rest[:0]


def fm_semantic_score(fm_sem, mask):
    """Landslide-likeness of every foundation-model cell, s = cos(f, mu_ls) - cos(f, mu_bg), where mu_ls / mu_bg
    are batch prototypes of the L2-normalised frozen features weighted by the landslide / background fraction
    of each cell. High s on background = semantic confuser (bare soil, river bed, road ...). No gradient."""
    with torch.no_grad():
        f = fm_sem.float()
        h, w = f.shape[-2:]
        t = F.adaptive_avg_pool2d(mask.float(), (h, w))
        mu_ls = F.normalize((f * t).sum(dim=(0, 2, 3)), dim=0)
        mu_bg = F.normalize((f * (1.0 - t)).sum(dim=(0, 2, 3)), dim=0)
        s = torch.einsum('bdhw,d->bhw', f, mu_ls - mu_bg)
    return s[:, None]


def pixel_contrast_loss(emb, seg_logits, mask, n_per_class=256, temperature=0.1, pos_thr=0.9, neg_thr=0.1,
                        mining='error', sem_score=None, sem_frac=0.2):
    """Supervised pixel contrastive loss on the 1/4-resolution embedding with hard-anchor mining (FP32).
    mining  'error'    : half of the anchors of each class are pixels the current prediction gets wrong;
            'fm'       : half are semantic hard pixels - the top sem_frac of background pixels the frozen
                         foundation model finds most landslide-like, and of landslide pixels it finds least so;
            'error+fm' : one third prediction errors, one third semantic hard pixels, one third random;
            'random'   : uniformly random anchors.
    Without sem_score, 'fm' / 'error+fm' fall back to 'error'."""
    emb = F.normalize(emb.float(), dim=1)
    _, D, h, w = emb.shape
    zero = emb.sum() * 0.0
    with torch.no_grad():
        t = F.adaptive_avg_pool2d(mask.float(), (h, w)).flatten()
        prob = F.adaptive_avg_pool2d(torch.sigmoid(seg_logits.detach().float()), (h, w)).flatten()
        pos, neg = t >= pos_thr, t <= neg_thr
        err_p, err_n = pos & (prob < 0.5), neg & (prob >= 0.5)
        use_sem = mining in ('fm', 'error+fm') and sem_score is not None
        sem_p = sem_n = pos
        if use_sem:
            s = F.interpolate(sem_score.float(), size=(h, w), mode='bilinear', align_corners=False).flatten()
            sem_p, sem_n = _top_frac(pos, -s, sem_frac), _top_frac(neg, s, sem_frac)
        if mining == 'random':
            none = torch.zeros_like(pos)
            idx_p, idx_n = _sample_idx(pos, none, n_per_class), _sample_idx(neg, none, n_per_class)
        elif use_sem and mining == 'fm':
            idx_p, idx_n = _sample_groups(pos, [sem_p], n_per_class), _sample_groups(neg, [sem_n], n_per_class)
        elif use_sem:
            idx_p = _sample_groups(pos, [err_p, sem_p], n_per_class)
            idx_n = _sample_groups(neg, [err_n, sem_n], n_per_class)
        else:
            idx_p, idx_n = _sample_idx(pos, err_p, n_per_class), _sample_idx(neg, err_n, n_per_class)
    if idx_p.numel() < 2 or idx_n.numel() < 2:
        return zero
    z = emb.permute(0, 2, 3, 1).reshape(-1, D)
    anchors = torch.cat([z[idx_p], z[idx_n]], 0)
    labels = torch.cat([torch.ones(idx_p.numel(), device=emb.device),
                        torch.zeros(idx_n.numel(), device=emb.device)])
    n = anchors.shape[0]
    eye = torch.eye(n, device=emb.device, dtype=torch.bool)
    sim = (anchors @ anchors.t() / temperature).masked_fill(eye, -1e9)
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pos_mask = (labels[:, None] == labels[None, :]) & ~eye
    loss = -(log_prob * pos_mask).sum(1) / pos_mask.sum(1).clamp(min=1)
    return loss.mean() + zero


def compute_loss(out, mask, a):
    """Returns (total_loss, vector [seg, aux4, aux8, bnd, con, dis]) in FP32."""
    zero = torch.zeros((), device=mask.device)
    l_seg = bce_dice(out['seg'], mask)
    total = l_seg
    l_a4 = l_a8 = l_bnd = l_con = l_dis = zero
    if 'aux4' in out:
        size = mask.shape[-2:]
        l_a4 = bce_dice(F.interpolate(out['aux4'].float(), size=size, mode='bilinear', align_corners=False), mask)
        l_a8 = bce_dice(F.interpolate(out['aux8'].float(), size=size, mode='bilinear', align_corners=False), mask)
        total = total + a.aux_weights[0] * l_a4 + a.aux_weights[1] * l_a8
    if 'bnd' in out:
        l_bnd = bce_dice(out['bnd'], boundary_target(mask))
        total = total + a.boundary_weight * l_bnd
    if 'emb' in out:
        mining = arg(a, 'hard_mining')
        sem = (fm_semantic_score(out['fm_sem'], mask)
               if mining in ('fm', 'error+fm') and 'fm_sem' in out else None)
        l_con = pixel_contrast_loss(out['emb'], out['seg'], mask, a.contrast_anchors, a.contrast_temp,
                                    mining=mining, sem_score=sem)
        total = total + a.contrast_weight * l_con
    if 'distill' in out and 'fm_sem' in out:
        pred = _resize(out['distill'].float(), out['fm_sem'].shape[-2:])
        l_dis = (1.0 - F.cosine_similarity(pred, out['fm_sem'].float(), dim=1)).mean()
        total = total + arg(a, 'distill_weight') * l_dis
    parts = torch.stack([l_seg.detach(), l_a4.detach(), l_a8.detach(), l_bnd.detach(), l_con.detach(),
                         l_dis.detach()])
    return total, parts


# ----------------------------------------------------------------------------------------
# optimisation helpers
# ----------------------------------------------------------------------------------------
class ModelEMA:
    """Exponential moving average of all floating-point parameters and buffers."""

    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.updates = decay, 0
        self._pairs = None

    def _build(self, model):
        esd, msd = self.module.state_dict(), model.state_dict()
        fe, fm, oe, om = [], [], [], []
        for k, v in esd.items():
            if v.dtype.is_floating_point:
                fe.append(v)
                fm.append(msd[k])
            else:
                oe.append(v)
                om.append(msd[k])
        self._pairs = (fe, fm, oe, om)

    @torch.no_grad()
    def update(self, model):
        if self._pairs is None:
            self._build(model)
        self.updates += 1
        d = min(self.decay, (1.0 + self.updates) / (10.0 + self.updates))
        fe, fm, oe, om = self._pairs
        if hasattr(torch, '_foreach_mul_') and hasattr(torch, '_foreach_add_'):
            torch._foreach_mul_(fe, d)  # fused multi-tensor kernels (fast path)
            torch._foreach_add_(fe, fm, alpha=1.0 - d)
        else:
            for e, m in zip(fe, fm):
                e.mul_(d).add_(m, alpha=1.0 - d)
        for e, m in zip(oe, om):
            e.copy_(m)

    def state_dict(self):
        return {'module': self.module.state_dict(), 'updates': self.updates}

    def load_state_dict(self, sd):
        self.module.load_state_dict(sd['module'])
        self.updates = int(sd['updates'])
        self._pairs = None


def build_optimizer(model, a):
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        enc = n.startswith('encoder.')
        no_decay = p.ndim <= 1 or n.endswith('.bias')
        groups.setdefault((enc, no_decay), []).append(p)
    param_groups = []
    for (enc, no_decay), params in groups.items():
        lr = a.enc_lr if enc else a.dec_lr
        param_groups.append({'params': params, 'lr': lr, 'base_lr': lr,
                             'weight_decay': 0.0 if no_decay else a.weight_decay,
                             'group_name': ('encoder' if enc else 'decoder') + ('_no_decay' if no_decay else '')})
    return torch.optim.AdamW(param_groups, lr=a.dec_lr, betas=(0.9, 0.999), eps=1e-8)


def lr_factor(step, total, warmup, min_ratio):
    if warmup > 0 and step < warmup:
        return 0.01 + 0.99 * step / warmup
    prog = min(1.0, max(0.0, (step - warmup) / max(1, total - warmup)))
    return min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * prog))


# ----------------------------------------------------------------------------------------
# evaluation
# ----------------------------------------------------------------------------------------
def metrics_from_counts(tp, fp, fn, tn):
    tp, fp, fn, tn = (float(v) for v in (tp, fp, fn, tn))
    n = tp + fp + fn + tn
    prec = tp / (tp + fp) if tp + fp > 0 else 0.0
    rec = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn > 0 else 0.0
    iou = tp / (tp + fp + fn) if tp + fp + fn > 0 else 0.0
    iou_bg = tn / (tn + fp + fn) if tn + fp + fn > 0 else 0.0
    f1_bg = 2 * tn / (2 * tn + fp + fn) if tn + fp + fn > 0 else 0.0
    oa = (tp + tn) / n if n > 0 else 0.0
    pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / (n * n) if n > 0 else 0.0
    kappa = (oa - pe) / (1.0 - pe) if pe < 1.0 else 0.0
    return {'precision': prec, 'recall': rec, 'f1': f1, 'iou': iou, 'iou_bg': iou_bg,
            'miou': 0.5 * (iou + iou_bg), 'f1_bg': f1_bg, 'mf1': 0.5 * (f1 + f1_bg), 'oa': oa,
            'kappa': kappa, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}


@torch.no_grad()
def predict_modes(model, x, modes, amp, channels_last, ms_scales=(0.75, 1.0, 1.25)):
    """x: Bx3xHxW float in [0,1]. Returns {mode: Bx1xHxW FP32 probabilities} for modes in
    {'noTTA', 'TTA', 'MSTTA'}, sharing forward passes: noTTA is the identity pass among the 8 D4 passes, TTA
    their mean, MSTTA the mean over scales of [TTA at scale 1, flip-averaged prediction at every other scale]
    (8 + 2 per extra scale forward passes)."""

    def run(inp):
        inp = inp.contiguous(memory_format=torch.channels_last) if channels_last else inp.contiguous()
        with torch.autocast(device_type=inp.device.type, dtype=torch.float16, enabled=amp):
            return torch.sigmoid(model(inp)['seg'].float())

    modes = set(modes)
    if not modes & {'TTA', 'MSTTA'}:
        return {'noTTA': run(x)}
    H, W = x.shape[-2:]
    acc = torch.zeros(x.shape[0], 1, H, W, device=x.device, dtype=torch.float32)
    ident = None
    for flip in (False, True):
        xf = torch.flip(x, dims=[3]) if flip else x
        for k in range(4):
            p = torch.rot90(run(torch.rot90(xf, k, dims=[2, 3])), -k, dims=[2, 3])
            p = torch.flip(p, dims=[3]) if flip else p
            if ident is None:
                ident = p
            acc += p
    d4 = acc / 8.0
    out = {}
    if 'noTTA' in modes:
        out['noTTA'] = ident
    if 'TTA' in modes:
        out['TTA'] = d4
    if 'MSTTA' in modes:
        ps = [d4]
        for s in ms_scales:
            if abs(float(s) - 1.0) < 1e-6:
                continue
            h, w = max(32, int(round(H * s / 32)) * 32), max(32, int(round(W * s / 32)) * 32)
            if (h, w) == (H, W):  # tiny inputs: the scale rounds back to the original size
                continue
            xs = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False, antialias=h < H)
            q = 0.5 * (run(xs) + torch.flip(run(torch.flip(xs, dims=[3])), dims=[3]))
            ps.append(F.interpolate(q, size=(H, W), mode='bilinear', align_corners=False))
        out['MSTTA'] = torch.stack(ps).mean(0)
    return out


def predict_probs(model, x, tta, amp, channels_last):
    """x: Bx3xHxW float in [0,1]. Returns Bx1xHxW FP32 probabilities (D4-TTA averaged if tta)."""
    mode = 'TTA' if tta else 'noTTA'
    return predict_modes(model, x, (mode,), amp, channels_last)[mode]


@torch.no_grad()
def evaluate(model, loader, d, a, modes=('noTTA',), thresholds=None, region_ids=None, n_regions=0, sink=None,
             sink_mode=None, per_tile=False):
    """Dataset-level confusion counts [tp, fp, fn, tn] for every prediction mode and threshold (and per region /
    per tile), summed over all ranks. thresholds: {mode: [t, ...]} (default [0.5]); per-tile counts use the last
    threshold of each mode. Returns {'counts': {mode: T x 4}, 'regions': {mode: R x T x 4} | None,
    'tiles': {mode: N x 4} | None} (CPU, float64)."""
    model.eval()
    modes = list(modes)
    ths = {m: list((thresholds or {}).get(m, [0.5])) for m in modes}
    dev = d.device
    counts = {m: torch.zeros(len(ths[m]), 4, dtype=torch.float64, device=dev) for m in modes}
    reg = ({m: torch.zeros(n_regions, len(ths[m]), 4, dtype=torch.float64, device=dev) for m in modes}
           if n_regions else None)
    tiles = ({m: torch.zeros(len(loader.dataset), 4, dtype=torch.float64, device=dev) for m in modes}
             if per_tile else None)
    ms_scales = tuple(float(s) for s in arg(a, 'ms_scales'))
    for img, mask, idx in loader:
        x = img.to(dev, non_blocking=True).float().div_(255.0)
        y = mask.to(dev, non_blocking=True) > 0
        probs = predict_modes(model, x, modes, a.amp, a.channels_last, ms_scales)
        B = x.shape[0]
        yf = y.reshape(B, -1)
        npx = float(yf.shape[1])
        rid = region_ids[idx].to(dev) if reg is not None else None
        for m in modes:
            pf = probs[m].reshape(B, -1)
            per = None
            for ti, t in enumerate(ths[m]):
                pr = pf >= t
                tp = (pr & yf).sum(1).double()
                fp = (pr & ~yf).sum(1).double()
                fn = (~pr & yf).sum(1).double()
                per = torch.stack([tp, fp, fn, npx - tp - fp - fn], 1)
                counts[m][ti] += per.sum(0)
                if reg is not None:
                    reg[m][:, ti].index_add_(0, rid, per)
            if tiles is not None:
                tiles[m][idx.to(dev)] = per
        if sink is not None:
            sink(idx, x, y, probs[sink_mode if sink_mode in probs else modes[-1]])
    for m in modes:
        d.sum_(counts[m])
        if reg is not None:
            d.sum_(reg[m])
        if tiles is not None:
            d.sum_(tiles[m])

    def cpu(dd):
        return None if dd is None else {k: v.cpu() for k, v in dd.items()}

    return {'counts': cpu(counts), 'regions': cpu(reg), 'tiles': cpu(tiles)}


class VisSink:
    """Writes RGB | ground truth | prediction-error panels (TP green, FP red, FN blue)."""

    def __init__(self, folder, rows, max_n, threshold, enabled):
        self.folder, self.rows, self.max_n, self.thr = Path(folder), rows, max_n, threshold
        self.enabled, self.n = enabled and max_n > 0, 0
        if self.enabled:
            self.folder.mkdir(parents=True, exist_ok=True)

    def __call__(self, idx, x, y, p):
        if not self.enabled or self.n >= self.max_n:
            return
        for b in range(x.shape[0]):
            if self.n >= self.max_n:
                return
            img = (x[b].permute(1, 2, 0).float().cpu().numpy() * 255.0 + 0.5).clip(0, 255).astype(np.uint8)
            gt = y[b, 0].cpu().numpy().astype(bool)
            pr = (p[b, 0] >= self.thr).cpu().numpy()
            gt_ov, err = img.copy(), img.copy()
            gt_ov[gt] = (0.45 * gt_ov[gt] + 0.55 * np.array([255, 255, 0])).astype(np.uint8)
            for m, col in ((gt & pr, (0, 255, 0)), (~gt & pr, (255, 0, 0)), (gt & ~pr, (0, 90, 255))):
                err[m] = (0.4 * err[m] + 0.6 * np.array(col)).astype(np.uint8)
            panel = np.concatenate([img, gt_ov, err], 1)
            ok, buf = cv2.imencode('.png', cv2.cvtColor(panel, cv2.COLOR_RGB2BGR))
            if ok:
                buf.tofile(str(self.folder / f'{safe_name(self.rows[int(idx[b])]["name"])}.png'))
            self.n += 1


def param_report(model):
    """Trainable parameters (incl. training-only heads), frozen foundation-model parameters (counted once even
    if shared by ensemble members) and the parameters actually used at inference."""
    nets = [m for m in model.modules() if isinstance(m, DualPriorUNet)]
    total = sum(p.numel() for p in model.parameters())
    heads = 0
    for net in nets:
        for name in TRAIN_ONLY_HEADS:
            mod = getattr(net, name, None)
            if isinstance(mod, nn.Module):
                heads += sum(p.numel() for p in mod.parameters())
    fm_infer, fm_train = {}, {}
    for net in nets:
        if net.fm_name:
            n = sum(p.numel() for p in get_foundation(net.fm_name, net.fm_pretrained).parameters())
            (fm_infer if net.fm_mode == 'fuse' else fm_train)[net.fm_name] = n
    fm_i = sum(fm_infer.values())
    return {'params_M': total / 1e6, 'train_only_heads_M': heads / 1e6, 'fm_frozen_params_M': fm_i / 1e6,
            'fm_training_only_params_M': sum(fm_train.values()) / 1e6,
            'inference_params_M': (total - heads + fm_i) / 1e6}


@torch.no_grad()
def model_stats(model, size, device, amp, channels_last=False):
    stats = param_report(model)
    model.eval()
    try:
        from torch.utils.flop_counter import FlopCounterMode
        x = torch.rand(1, 3, size, size, device=device)
        with FlopCounterMode(display=False) as fc:
            model(x)
        flops = fc.get_total_flops()
        stats['gflops'] = flops / 1e9
        stats['gmacs'] = flops / 2e9
    except Exception as e:  # noqa: BLE001
        stats['flops_error'] = str(e)[:200]
    if device.type == 'cuda':
        x = torch.rand(8, 3, size, size, device=device)
        if channels_last:
            x = x.contiguous(memory_format=torch.channels_last)
        for _ in range(3):
            with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
                model(x)
        torch.cuda.synchronize(device)
        t0, n_it = time.time(), 10
        for _ in range(n_it):
            with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
                model(x)
        torch.cuda.synchronize(device)
        stats['fps_batch8'] = 8 * n_it / (time.time() - t0)
        stats['gpu'] = torch.cuda.get_device_name(device)
    return stats


BENCHMARKS = {
    # (method, source, same split?, P, R, IoU, mIoU, OA, mF1, F1)  values in %; F1 = landslide-class F1
    'SAT': [
        ('FCN (VGG16)', 'Xu et al. 2024, Sci. Data', 'no', 62.981, 84.142, 55.716, 75.173, 94.972, 84.391, None),
        ('U-Net (ResNet50)', 'Xu et al. 2024, Sci. Data', 'no', 61.795, 78.550, 51.179, 72.619, 94.410, 82.316, None),
        ('DeepLabv3+ (ResNet50)', 'Xu et al. 2024, Sci. Data', 'no', 74.275, 89.187, 68.137, 82.397, 96.881, 89.675, None),
        ('MFFENet (ResNet50)', 'Xu et al. 2024, Sci. Data', 'no', 74.141, 89.141, 67.998, 82.318, 96.862, 89.621, None),
        ('SegFormer', '2025, quoted by SC-Net', 'no', None, None, 79.30, None, None, None, None),
        ('U-Net (ResNet50)', 'Song et al. 2026 (SC-Net paper)', 'yes', 84.39, 88.02, 75.69, None, None, None, 86.17),
        ('DeepLabV3+ (ResNet50)', 'Song et al. 2026 (SC-Net paper)', 'yes', 86.90, 87.99, 77.68, None, None, None, 87.44),
        ('SC-Net (RegNetY-6.4GF)', 'Song et al. 2026, Remote Sens.', 'yes', 87.77, 89.74, 79.76, None, None, None, 88.74),
    ],
    'UAV': [
        ('FCN (VGG16)', 'Xu et al. 2024, Sci. Data', 'no', 75.045, 84.016, 65.057, 77.456, 91.468, 86.724, None),
        ('U-Net (ResNet50)', 'Xu et al. 2024, Sci. Data', 'no', 73.694, 86.394, 65.991, 78.019, 91.658, 87.136, None),
        ('DeepLabv3+ (ResNet50)', 'Xu et al. 2024, Sci. Data', 'no', 89.289, 93.739, 84.261, 90.142, 96.721, 94.715, None),
        ('MFFENet (ResNet50)', 'Xu et al. 2024, Sci. Data', 'no', 89.326, 93.839, 84.375, 90.214, 96.746, 94.756, None),
        ('SegFormer', '2025, quoted by SC-Net', 'no', None, None, 87.80, None, None, None, None),
        ('U-Net (ResNet50)', 'Song et al. 2026 (SC-Net paper)', 'no', 91.78, 91.80, 84.82, None, None, None, 91.79),
        ('DeepLabV3+ (ResNet50)', 'Song et al. 2026 (SC-Net paper)', 'no', 92.76, 93.92, 87.51, None, None, None, 93.34),
        ('SC-Net (RegNetY-6.4GF)', 'Song et al. 2026, Remote Sens.', 'no', 94.82, 94.51, 89.89, None, None, None, 94.67),
    ],
}


def fmt(v):
    return '-' if v is None else f'{v:.2f}'


def describe_semantic_prior(margs):
    fm = arg(margs, 'fm')
    if not fm_enabled(fm):
        return 'none'
    mode = arg(margs, 'fm_mode')
    return f'`{fm}` ({mode}' + (f', {arg(margs, "fm_fusion")} fusion)' if mode == 'fuse' else ', training only)')


def results_markdown(res, subset):
    margs = res.get('model_args') or res['args']
    main_mode = res.get('tta_mode', 'TTA' if res.get('tta') else 'noTTA')
    lines = [f'# Results - {res["run"]}', '',
             f'Encoder `{arg(margs, "encoder")}`; semantic prior: {describe_semantic_prior(margs)}; '
             f'anchor mining: {arg(margs, "hard_mining")}; subset **{subset}**; test tiles: {res["n_test"]}; '
             f'selected weights: {res["weights"]}; main setting: {MODE_LABELS[main_mode]} at decision threshold '
             f't* = {res["threshold"]:.3f} ({res["threshold_source"]}).', '',
             'All values in %. F1 / IoU / P / R are for the **landslide** class (dataset-level pixel counts). '
             'mF1 = mean of landslide and background F1 (the "F1 score" reported by Xu et al. 2024); '
             'mIoU = mean of landslide and background IoU.', '',
             '| Method | Source | Same test split | P | R | F1 | IoU | mIoU | OA | mF1 | Kappa |',
             '|---|---|---|---|---|---|---|---|---|---|---|']
    if res.get('tta'):
        ours = [('test_noTTA@0.5', 'no TTA, t=0.5'), ('test_TTA@0.5', 'D4-TTA, t=0.5'), ('test_TTA@t*', 'D4-TTA, t=t*'),
                ('test_MSTTA@0.5', 'D4+MS-TTA, t=0.5'), ('test_MSTTA@t*', 'D4+MS-TTA, t=t*')]
    else:
        ours = [('test_noTTA@0.5', 'no TTA, t=0.5'), ('test_noTTA@t*', 'no TTA, t=t*')]
    for key, label in ours:
        m = res.get(key)
        if m is None:
            continue
        label = f'Ours, {label}' + (' (main)' if key == res.get('main_key') else '')
        lines.append(f'| **{label}** | this work | yes | {100 * m["precision"]:.2f} | {100 * m["recall"]:.2f} | '
                     f'{100 * m["f1"]:.2f} | **{100 * m["iou"]:.2f}** | {100 * m["miou"]:.2f} | {100 * m["oa"]:.2f} | '
                     f'{100 * m["mf1"]:.2f} | {100 * m["kappa"]:.2f} |')
    for (name, src, same, P, R, iou, miou, oa, mf1, f1) in BENCHMARKS.get(subset, []):
        if f1 is None and P is not None and R is not None:
            f1 = 2 * P * R / (P + R)
        if subset == 'UAV':
            same = 'no'
        lines.append(f'| {name} | {src} | {same} | {fmt(P)} | {fmt(R)} | {fmt(f1)} | {fmt(iou)} | {fmt(miou)} | '
                     f'{fmt(oa)} | {fmt(mf1)} | - |')
    lines += ['', 'Notes: Xu et al. (2024) and SegFormer used their own random 7:3 splits. For SAT, the SC-Net '
              'rows use the identical public test split; SC-Net selected its checkpoint on that test split, '
              'whereas this work selects on a validation split carved from the training tiles. '
              'For UAV no public split exists; our seeded split is written by prepare_data.py.', '']
    if res.get('per_region'):
        lines += ['## Per-region test metrics (main setting)', '',
                  '| Region | tiles | P | R | F1 | IoU |', '|---|---|---|---|---|---|']
        for rg, m in res['per_region'].items():
            lines.append(f'| {rg} | {m["tiles"]} | {100 * m["precision"]:.2f} | {100 * m["recall"]:.2f} | '
                         f'{100 * m["f1"]:.2f} | {100 * m["iou"]:.2f} |')
        lines.append('')
    val_modes = [m for m in ('TTA', 'MSTTA', 'noTTA') if res.get(f'val_{m}@0.5')]
    if not val_modes and res.get('val_at_0.5'):  # results written by the SP-UNet version
        res = {**res, f'val_{main_mode}@0.5': res['val_at_0.5'], f'val_{main_mode}@t*': res.get('val_at_t*')}
        val_modes = [main_mode]
    for m in val_modes:
        v, vt = res[f'val_{m}@0.5'], res.get(f'val_{m}@t*') or res[f'val_{m}@0.5']
        t_m = (res.get('thresholds') or {}).get(m, res['threshold'])
        lines.append(f'Validation ({res["n_val"]} tiles, {MODE_LABELS[m]}): IoU {100 * v["iou"]:.2f} / '
                     f'F1 {100 * v["f1"]:.2f} at t=0.5; IoU {100 * vt["iou"]:.2f} / F1 {100 * vt["f1"]:.2f} '
                     f'at t*={t_m:.3f}.')
    if val_modes:
        lines.append('')
    st = res.get('model_stats') or {}
    if st:
        size = arg(margs, 'img_size')
        txt = (f'Model: {st.get("params_M", 0):.2f} M trainable parameters (incl. '
               f'{st.get("train_only_heads_M", 0):.2f} M in training-only heads)')
        if st.get('fm_frozen_params_M'):
            txt += f' + {st["fm_frozen_params_M"]:.1f} M frozen foundation-model parameters'
        if st.get('fm_training_only_params_M'):
            txt += f' (+ {st["fm_training_only_params_M"]:.1f} M frozen foundation-model parameters used only in training)'
        if 'inference_params_M' in st:
            txt += f'; {st["inference_params_M"]:.2f} M parameters used at inference'
        txt += (f'; {st.get("gmacs", float("nan")):.1f} GMACs / {st.get("gflops", float("nan")):.1f} GFLOPs '
                f'at {size}x{size}')
        if 'fps_batch8' in st:
            txt += f', {st["fps_batch8"]:.1f} img/s (batch 8, FP16, {st.get("gpu", "")}, single pass)'
        lines += [txt + '.', '']
    return '\n'.join(lines)


def _ths(t):
    return [0.5] if abs(float(t) - 0.5) < 1e-9 else [0.5, float(t)]


def write_per_tile(path, rows, tiles):
    """Per-tile confusion counts of every evaluated mode at its own t* (input of paired significance tests)."""
    modes = list(tiles)
    path = Path(path)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['name', 'region'] + [f'{m}_{k}' for m in modes for k in ('tp', 'fp', 'fn', 'tn')])
        for i, r in enumerate(rows):
            w.writerow([r['name'], r['region']] + [int(round(float(tiles[m][i, j]))) for m in modes for j in range(4)])
    os.replace(tmp, path)


def final_evaluation(a, d, model, rows, loaders, out_dir, info):
    """TTA mode and threshold chosen on validation (if allowed); test with no TTA, D4-TTA and (--ms-tta)
    D4+MS-TTA in one pass; per-region and per-tile counts; results.json / results.md / per_tile_test.csv."""
    use_tta = not a.no_tta
    ms = use_tta and bool(arg(a, 'ms_tta'))
    sel_modes = (['TTA'] + (['MSTTA'] if ms else [])) if use_tta else ['noTTA']
    regions = sorted({r['region'] for r in rows['test']})
    rid_map = {rg: i for i, rg in enumerate(regions)}
    test_rids = torch.tensor([rid_map[r['region']] for r in rows['test']], dtype=torch.long)
    res = {'run': out_dir.name, 'tag': arg(a, 'tag') or out_dir.name, 'args': vars(a),
           'n_test': len(rows['test']), 'n_val': len(rows['val']), **info}
    thr = {m: float(a.threshold) for m in sel_modes}
    thr_src, val_best = 'fixed (--threshold)', {}
    if loaders.get('val') is not None and info.get('select_threshold', False):
        ev = evaluate(model, loaders['val'], d, a, modes=sel_modes,
                      thresholds={m: SWEEP_THRESHOLDS for m in sel_modes})
        i05 = SWEEP_THRESHOLDS.index(0.5)
        for m in sel_modes:
            c = ev['counts'][m]
            sweep = [metrics_from_counts(*c[i].tolist()) for i in range(len(SWEEP_THRESHOLDS))]
            best = max(range(len(SWEEP_THRESHOLDS)),
                       key=lambda i: (round(sweep[i]['iou'], 6), -abs(SWEEP_THRESHOLDS[i] - 0.5)))
            thr[m], val_best[m] = float(SWEEP_THRESHOLDS[best]), sweep[best]['iou']
            res[f'val_sweep_{m}'] = {f'{t:.3f}': {'iou': s['iou'], 'f1': s['f1']}
                                     for t, s in zip(SWEEP_THRESHOLDS, sweep)}
            res[f'val_{m}@0.5'], res[f'val_{m}@t*'] = sweep[i05], sweep[best]
        m0 = sel_modes[0]
        res['val_sweep'], res['val_at_0.5'], res['val_at_t*'] = (res[f'val_sweep_{m0}'], res[f'val_{m0}@0.5'],
                                                                 res[f'val_{m0}@t*'])
        thr_src = 'best validation IoU' + (' with TTA' if use_tta else '')
    main = sel_modes[0]
    if ms and val_best and val_best['MSTTA'] > val_best['TTA']:
        main = 'MSTTA'
    if ms:
        res['tta_mode_source'] = ('D4+MS-TTA beat D4-TTA on validation' if main == 'MSTTA' else
                                  'D4-TTA >= D4+MS-TTA on validation' if val_best else
                                  'D4-TTA (no validation split to choose on)')
    res['threshold'], res['threshold_source'], res['thresholds'] = thr[main], thr_src, dict(thr)
    log(d, f'[eval] main setting {MODE_LABELS[main]}, decision threshold t* = {thr[main]:.3f} ({thr_src})')
    test_modes = (['noTTA'] if use_tta else []) + sel_modes
    ths = {m: _ths(thr.get(m, thr[sel_modes[0]])) for m in test_modes}  # no-TTA rows use the TTA threshold
    sink = VisSink(out_dir / 'vis_test', rows['test'], a.save_vis, thr[main], d.main)
    ev = evaluate(model, loaders['test'], d, a, modes=test_modes, thresholds=ths, region_ids=test_rids,
                  n_regions=len(regions), sink=sink, sink_mode=main, per_tile=True)
    for m in test_modes:
        c = ev['counts'][m]
        res[f'test_{m}@0.5'] = metrics_from_counts(*c[0].tolist())
        res[f'test_{m}@t*'] = metrics_from_counts(*c[-1].tolist())
    res['tta'], res['tta_mode'] = use_tta, main
    res['main_key'] = f'test_{main}@t*'
    res['main'] = res[res['main_key']]
    res['ablation_key'] = f'test_{sel_modes[0]}@t*'  # the same setting for every ablation row
    n_tiles = {rg: sum(1 for r in rows['test'] if r['region'] == rg) for rg in regions}
    res['per_region_by_mode'] = {}
    for m in test_modes:
        pr = {}
        for rg, i in rid_map.items():
            pr[rg] = metrics_from_counts(*ev['regions'][m][i, -1].tolist())
            pr[rg]['tiles'] = int(n_tiles[rg])
        res['per_region_by_mode'][m] = pr
    res['per_region'] = res['per_region_by_mode'][main]
    if d.main:
        write_per_tile(out_dir / 'per_tile_test.csv', rows['test'], ev['tiles'])
        res['per_tile_file'] = 'per_tile_test.csv'
        try:
            res['model_stats'] = model_stats(model, a.img_size, d.device, a.amp, a.channels_last)
        except Exception as e:  # noqa: BLE001
            res['model_stats'] = {'error': str(e)[:200]}
        subset = info.get('subset', 'SAT')
        write_json(res, out_dir / 'results.json')
        md = results_markdown(res, subset)
        (out_dir / 'results.md').write_text(md, encoding='utf-8')
        print('\n' + md, flush=True)
    d.barrier()
    return res


# ----------------------------------------------------------------------------------------
# main
# ----------------------------------------------------------------------------------------
def parse_args(argv=None):
    p = argparse.ArgumentParser(description='DPF-Net (Dual-Prior Fusion Network) for CAS Landslide segmentation')
    p.add_argument('--splits', required=True, help='folder with train/val/trainval/test.csv (prepare_data.py)')
    p.add_argument('--out', required=True, help='output folder of this run')
    g = p.add_argument_group('model')
    g.add_argument('--encoder', default='convnext_tiny.dinov3_lvd1689m')
    g.add_argument('--no-pretrained', action='store_true', help='random encoder init (smoke tests only)')
    g.add_argument('--no-hires', action='store_true', help='drop the high-resolution stem (implies --no-prior)')
    g.add_argument('--no-prior', action='store_true', help='drop the spectral-structural prior')
    g.add_argument('--no-attention', action='store_true', help='drop scSE attention in the decoder')
    g.add_argument('--boundary-weight', type=float, default=0.3, help='0 disables the boundary head')
    g.add_argument('--contrast-weight', type=float, default=0.1, help='0 disables the contrastive head')
    g.add_argument('--contrast-temp', type=float, default=0.1)
    g.add_argument('--contrast-anchors', type=int, default=256, help='anchors per class per GPU')
    g.add_argument('--aux-weights', type=float, nargs=2, default=[0.4, 0.2], help='deep supervision 1/4, 1/8')
    g.add_argument('--grad-ckpt', action='store_true', help='gradient checkpointing in the encoder')
    g = p.add_argument_group('foundation-model semantic prior')
    g.add_argument('--fm', default='vit_large_patch16_dinov3.sat493m',
                   help="frozen timm ViT used as semantic prior ('none' = SP-UNet), e.g. "
                        'vit_large_patch16_dinov3.lvd1689m (web images) or vit_base_patch16_dinov3.lvd1689m')
    g.add_argument('--fm-mode', choices=['fuse', 'distill'], default='fuse',
                   help='fuse: gated injection of the FM features (FM needed at inference); distill: FM used only '
                        'during training (feature distillation + semantic mining), no inference cost')
    g.add_argument('--fm-fusion', choices=['gated', 'add'], default='gated',
                   help='gated: structure-aware gated injection; add: plain residual addition (ablation)')
    g.add_argument('--fm-size', type=int, default=0,
                   help='FM input size for img-size tiles (0 = img-size; rounded to the patch size), e.g. 384')
    g.add_argument('--fm-layers', type=int, default=4, help='evenly spaced ViT blocks whose features are used')
    g.add_argument('--fm-channels', type=int, default=256, help='width of the FM adapter')
    g.add_argument('--hard-mining', choices=['auto', 'error', 'fm', 'error+fm', 'random'], default='auto',
                   help="anchor mining of the pixel contrastive loss; auto = 'error+fm' with a foundation model, "
                        "otherwise 'error'")
    g.add_argument('--distill-weight', type=float, default=0.5, help='feature-distillation weight (--fm-mode distill)')
    g = p.add_argument_group('training')
    g.add_argument('--img-size', type=int, default=512)
    g.add_argument('--epochs', type=int, default=80)
    g.add_argument('--batch-size', type=int, default=8, help='per GPU')
    g.add_argument('--workers', type=int, default=2, help='DataLoader workers per process')
    g.add_argument('--enc-lr', type=float, default=1e-4)
    g.add_argument('--dec-lr', type=float, default=4e-4)
    g.add_argument('--weight-decay', type=float, default=0.05)
    g.add_argument('--warmup-epochs', type=float, default=2.0)
    g.add_argument('--min-lr-ratio', type=float, default=0.01)
    g.add_argument('--ema-decay', type=float, default=0.9995)
    g.add_argument('--clip-grad', type=float, default=1.0)
    g.add_argument('--no-amp', action='store_true')
    g.add_argument('--no-channels-last', action='store_true')
    g.add_argument('--seed', type=int, default=42)
    g.add_argument('--full-train', action='store_true',
                   help='train on train+val with a fixed schedule; evaluate the final EMA weights at --threshold')
    g.add_argument('--threshold', type=float, default=0.5, help='decision threshold when not selected on val')
    g.add_argument('--max-train-hours', type=float, default=10.5, help='stop + checkpoint before this budget')
    g.add_argument('--resume', default='', help='last.pth of a previous (possibly interrupted) run')
    g.add_argument('--val-every', type=int, default=1)
    g.add_argument('--track-test', action='store_true',
                   help='also log test IoU every epoch (reporting only - never used for selection)')
    g.add_argument('--log-every', type=int, default=50)
    g.add_argument('--tag', default='', help='experiment id stored in results.json (default: the --out folder name)')
    g.add_argument('--no-eval-on-timeout', action='store_true',
                   help='if the time budget stops training, save last.pth and skip the final evaluation '
                        '(used by ablation.py; the run is resumed in the next session)')
    g.add_argument('--stop-after-epochs', type=int, default=0,
                   help='testing: behave as if the time budget ran out after N epochs of this invocation')
    g = p.add_argument_group('augmentation')
    g.add_argument('--aug-affine-p', type=float, default=0.5)
    g.add_argument('--aug-rotate', type=float, default=30.0)
    g.add_argument('--aug-scale-min', type=float, default=0.75)
    g.add_argument('--aug-scale-max', type=float, default=1.33)
    g.add_argument('--aug-translate', type=float, default=0.1)
    g.add_argument('--aug-color-p', type=float, default=0.8)
    g.add_argument('--aug-brightness', type=float, default=0.2)
    g.add_argument('--aug-contrast', type=float, default=0.2)
    g.add_argument('--aug-saturation', type=float, default=0.2)
    g.add_argument('--aug-channel-gain', type=float, default=0.05)
    g.add_argument('--aug-gamma-p', type=float, default=0.3)
    g.add_argument('--aug-noise-p', type=float, default=0.2)
    g.add_argument('--aug-noise-std', type=float, default=0.02)
    g = p.add_argument_group('evaluation')
    g.add_argument('--no-tta', action='store_true')
    g.add_argument('--ms-tta', action='store_true',
                   help='also evaluate D4 + multi-scale TTA; the main setting (D4 or D4+MS) is chosen on validation')
    g.add_argument('--ms-scales', type=float, nargs='+', default=[0.75, 1.0, 1.25])
    g.add_argument('--eval-only', action='store_true', help='skip training; evaluate --eval-ckpts (or <out>/best.pth)')
    g.add_argument('--eval-ckpts', nargs='*', default=[], help='several checkpoints = probability ensemble')
    g.add_argument('--save-vis', type=int, default=24, help='qualitative test panels to save (rank 0)')
    g.add_argument('--debug-limit', type=int, default=0, help='use only the first N tiles of every split (tests)')
    a = p.parse_args(argv)
    if a.no_hires and not a.no_prior:
        a.no_prior = True
    a.fm = 'none' if str(a.fm).strip().lower() in ('', 'none', 'null', 'off', 'no') else str(a.fm).strip()
    if a.hard_mining == 'auto':
        a.hard_mining = 'error+fm' if fm_enabled(a.fm) else 'error'
    elif a.hard_mining in ('fm', 'error+fm') and not fm_enabled(a.fm):
        p.error(f'--hard-mining {a.hard_mining} needs a foundation model (--fm)')
    if a.fm_mode == 'distill' and not fm_enabled(a.fm):
        p.error('--fm-mode distill needs a foundation model (--fm)')
    if a.fm_size < 0 or a.fm_layers < 1 or a.fm_channels < 8:
        p.error('--fm-size must be >= 0, --fm-layers >= 1, --fm-channels >= 8')
    a.amp = (not a.no_amp) and torch.cuda.is_available()
    a.channels_last = (not a.no_channels_last) and torch.cuda.is_available()
    return a


def load_rows(a):
    sp = Path(a.splits)
    rows = {'train': read_split(sp / ('trainval.csv' if a.full_train else 'train.csv')),
            'val': [] if a.full_train or not (sp / 'val.csv').exists() else read_split(sp / 'val.csv'),
            'test': read_split(sp / 'test.csv')}
    if a.debug_limit:
        rows = {k: v[:a.debug_limit] for k, v in rows.items()}
    subset = 'SAT'
    if (sp / 'summary.json').exists():
        subset = json.loads((sp / 'summary.json').read_text(encoding='utf-8')).get('subset', 'SAT')
    return rows, subset


def make_eval_loader(rows, a, d, persistent=False):
    if not rows:
        return None
    ds = SegDataset(rows, a.img_size, train=False)
    return DataLoader(ds, batch_size=a.batch_size, sampler=ShardSampler(len(ds), d.rank, d.world),
                      num_workers=a.workers, pin_memory=d.device.type == 'cuda', drop_last=False,
                      persistent_workers=persistent and a.workers > 0)


def eval_only(a, d, rows, subset, out_dir):
    ckpts = a.eval_ckpts or [str(out_dir / 'best.pth')]
    models, full_train_any, names, member_args = [], False, [], []
    for c in ckpts:
        sd = torch.load(c, map_location='cpu', weights_only=True)
        margs = argparse.Namespace(**sd['args'])
        if margs.img_size != a.img_size:
            log(d, f'WARNING: {c} was trained at {margs.img_size}px, evaluating at {a.img_size}px')
        m = build_model(margs, pretrained=False)
        m.load_state_dict(sd['model'])
        models.append(m)
        fm = arg(margs, 'fm')
        names.append(f'{Path(c).parent.name}/{Path(c).name} ({margs.encoder}'
                     + (f' + {fm} [{arg(margs, "fm_mode")}]' if fm_enabled(fm) else '') + ')')
        member_args.append({**{k: LEGACY_DEFAULTS[k] for k in LEGACY_DEFAULTS if k not in sd['args']}, **sd['args']})
        full_train_any |= bool(getattr(margs, 'full_train', False))
    model = models[0] if len(models) == 1 else Ensemble(models)
    model = model.to(d.device).eval()
    if a.channels_last:
        model = model.to(memory_format=torch.channels_last)
    loaders = {'val': make_eval_loader(rows['val'], a, d), 'test': make_eval_loader(rows['test'], a, d)}
    info = {'subset': subset, 'weights': 'ensemble of ' + ', '.join(names) if len(names) > 1 else names[0],
            'select_threshold': not full_train_any and loaders['val'] is not None, 'schedule_complete': None,
            'eval_only': True, 'checkpoints': [str(c) for c in ckpts]}
    if len(member_args) == 1:
        info['model_args'] = member_args[0]  # the trained model's settings (results 'args' = this invocation)
    else:
        info['member_args'] = member_args
    log(d, f'[eval-only] {info["weights"]}')
    final_evaluation(a, d, model, rows, loaders, out_dir, info)
    if d.main:
        write_json({'tag': a.tag or out_dir.name, 'schedule_complete': None, 'evaluated': True, 'eval_only': True},
                   out_dir / 'status.json')


def train(a, d, rows, subset, out_dir):
    t_start = time.time()
    has_val = len(rows['val']) > 0
    split_h = rows_hash(rows['train'], rows['val'], rows['test'])

    # ---- model (rank 0 downloads the pretrained weights first) ----
    model = None
    if d.main:
        model = build_model(a, pretrained=not a.no_pretrained)
    d.barrier()
    if not d.main:
        model = build_model(a, pretrained=not a.no_pretrained)
    if d.enabled and d.device.type == 'cuda':
        model = nn.SyncBatchNorm.convert_sync_batchnorm(model)
    model = model.to(d.device)
    if a.channels_last:
        model = model.to(memory_format=torch.channels_last)
    ema = ModelEMA(model, a.ema_decay)
    optimizer = build_optimizer(model, a)
    scaler = torch.amp.GradScaler(d.device.type, enabled=a.amp)

    # ---- data ----
    ds_train = SegDataset(rows['train'], a.img_size, train=True)
    sampler = DistributedSampler(ds_train, num_replicas=d.world, rank=d.rank, shuffle=True, seed=a.seed,
                                 drop_last=True)
    dl_train = DataLoader(ds_train, batch_size=a.batch_size, sampler=sampler, num_workers=a.workers,
                          pin_memory=d.device.type == 'cuda', drop_last=True, persistent_workers=a.workers > 0)
    dl_val = make_eval_loader(rows['val'], a, d, persistent=True) if has_val else None
    dl_test_track = make_eval_loader(rows['test'], a, d, persistent=True) if a.track_test else None
    steps_per_epoch = len(dl_train)
    if steps_per_epoch == 0:
        raise SystemExit('Training split smaller than one global batch.')
    total_steps = a.epochs * steps_per_epoch
    warmup_steps = min(int(a.warmup_epochs * steps_per_epoch), max(total_steps - 1, 0))

    # ---- resume ----
    start_epoch, step, best_iou, best_epoch, history, prev_times = 0, 0, -1.0, 0, [], []
    if a.resume:
        ck = torch.load(a.resume, map_location='cpu', weights_only=True)
        old = ck['args']
        for k in ARCH_KEYS + ('full_train', 'hard_mining'):
            if arg(old, k) != arg(a, k):
                raise SystemExit(f'--resume: argument "{k}" differs ({arg(old, k)} vs {arg(a, k)}).')
        for k in ('boundary_weight', 'contrast_weight'):
            if (old.get(k, 0) > 0) != (getattr(a, k) > 0):
                raise SystemExit(f'--resume: head "{k}" enabled/disabled differently from the checkpoint.')
        if (any(w > 0 for w in old.get('aux_weights', [0, 0]))) != any(w > 0 for w in a.aux_weights):
            raise SystemExit('--resume: deep supervision enabled/disabled differently from the checkpoint.')
        if ck.get('split_hash') != split_h:
            log(d, 'WARNING: the data split differs from the checkpoint run (split hash mismatch).')
        if old.get('epochs') != a.epochs or old.get('steps_per_epoch') not in (None, steps_per_epoch):
            log(d, 'WARNING: epochs / steps-per-epoch changed - the LR schedule continues from the saved step.')
        model.load_state_dict(ck['model'])
        ema.load_state_dict(ck['ema'])
        optimizer.load_state_dict(ck['optimizer'])
        for gr in optimizer.param_groups:  # keep the base LRs of this invocation
            gr['base_lr'] = a.enc_lr if gr.get('group_name', '').startswith('encoder') else a.dec_lr
        if a.amp and ck.get('scaler'):
            scaler.load_state_dict(ck['scaler'])
        start_epoch, step = int(ck['epoch']), int(ck['step'])
        best_iou, best_epoch = float(ck['best_iou']), int(ck['best_epoch'])
        history = list(ck.get('history', []))
        prev_times = [float(h['seconds']) for h in history[-3:] if h.get('seconds')]
        if d.main:
            prev_best = Path(a.resume).parent / 'best.pth'
            if prev_best.exists() and not (out_dir / 'best.pth').exists():
                shutil.copy2(prev_best, out_dir / 'best.pth')
        log(d, f'[resume] {a.resume}: continuing at epoch {start_epoch + 1}/{a.epochs} (step {step}), '
               f'best val IoU {100 * best_iou:.2f} @ epoch {best_epoch}')
        del ck

    net = model
    if d.enabled:
        net = nn.parallel.DistributedDataParallel(
            model, device_ids=[d.local_rank] if d.device.type == 'cuda' else None,
            broadcast_buffers=False, find_unused_parameters=False, gradient_as_bucket_view=True)

    pr = param_report(model)
    log(d, f'[train] encoder {a.encoder} | {pr["params_M"]:.2f} M trainable params | hires={not a.no_hires} '
           f'prior={not a.no_prior} boundary={a.boundary_weight} contrast={a.contrast_weight} '
           f'(mining {a.hard_mining}) aux={a.aux_weights}')
    log(d, f'[train] semantic prior: {describe_semantic_prior(a)}'
           + (f' | {pr["fm_frozen_params_M"] + pr["fm_training_only_params_M"]:.1f} M frozen parameters, '
              f'blocks {model.fm_idx}, input scale {model.fm_scale:g}' if model.fm_name else ''))
    log(d, f'[train] tiles train/val/test = {len(rows["train"])}/{len(rows["val"])}/{len(rows["test"])} | '
           f'world {d.world} x batch {a.batch_size} = {d.world * a.batch_size} | {steps_per_epoch} steps/epoch | '
           f'{a.epochs} epochs | AMP {a.amp} | channels_last {a.channels_last} | device {d.device}')

    epoch_times, stopped_early = list(prev_times), False  # previous session's epoch times seed the time guard
    names = ['loss', 'seg', 'aux4', 'aux8', 'bnd', 'con', 'dis']
    for epoch in range(start_epoch, a.epochs):
        elapsed = time.time() - t_start
        est = float(np.mean(epoch_times[-3:])) if epoch_times else 0.0
        if d.any_flag(elapsed + 1.05 * est > a.max_train_hours * 3600):
            log(d, f'[train] time budget ({a.max_train_hours} h) reached before epoch {epoch + 1}; '
                   'stopping. Resume later with --resume <out>/last.pth')
            stopped_early = True
            break
        t_ep = time.time()
        sampler.set_epoch(epoch)
        net.train()
        acc = torch.zeros(len(names), device=d.device, dtype=torch.float64)
        n_ok = torch.zeros((), device=d.device, dtype=torch.float64)
        n_skip = 0
        for it, (img, mask, _) in enumerate(dl_train):
            x = img.to(d.device, non_blocking=True).float().div_(255.0)
            y = mask.to(d.device, non_blocking=True).float()
            x, y = gpu_augment(x, y, a)
            if a.channels_last:
                x = x.contiguous(memory_format=torch.channels_last)
            f = lr_factor(step, total_steps, warmup_steps, a.min_lr_ratio)
            for gr in optimizer.param_groups:
                gr['lr'] = gr['base_lr'] * f
            with torch.autocast(device_type=d.device.type, dtype=torch.float16, enabled=a.amp):
                out = net(x)
            loss, parts = compute_loss(out, y, a)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if a.clip_grad > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), a.clip_grad)
            scale_before = scaler.get_scale() if a.amp else 1.0
            scaler.step(optimizer)
            scaler.update()
            if a.amp and scaler.get_scale() < scale_before:
                n_skip += 1
            ema.update(model)
            step += 1
            vals = torch.cat([loss.detach().reshape(1), parts]).double()
            ok = torch.isfinite(vals).all()
            acc += torch.where(ok, vals, torch.zeros_like(vals))
            n_ok += ok.double()
            if d.main and a.log_every and (it + 1) % a.log_every == 0:
                m = (acc / n_ok.clamp(min=1)).tolist()
                print(f'  ep {epoch + 1} it {it + 1}/{steps_per_epoch} loss {m[0]:.4f} (seg {m[1]:.4f}) '
                      f'lr {optimizer.param_groups[0]["lr"]:.2e} | {(time.time() - t_ep) / (it + 1):.2f} s/it',
                      flush=True)
        d.sum_(acc)
        d.sum_(n_ok)
        tr = {k: v for k, v in zip(names, (acc / n_ok.clamp(min=1)).tolist())}

        val_m = None
        if dl_val is not None and ((epoch + 1) % a.val_every == 0 or epoch + 1 == a.epochs):
            c = evaluate(ema.module, dl_val, d, a, modes=('noTTA',))['counts']['noTTA']
            val_m = metrics_from_counts(*c[0].tolist())
        test_m = None
        if dl_test_track is not None:
            c = evaluate(ema.module, dl_test_track, d, a, modes=('noTTA',))['counts']['noTTA']
            test_m = metrics_from_counts(*c[0].tolist())
        improved = val_m is not None and val_m['iou'] > best_iou
        if improved:
            best_iou, best_epoch = val_m['iou'], epoch + 1
        epoch_times.append(time.time() - t_ep)
        rec = {'epoch': epoch + 1, 'lr_dec': optimizer.param_groups[0]['lr'], 'train': tr,
               'amp_skipped_steps': n_skip, 'amp_scale': scaler.get_scale() if a.amp else None,
               'val': val_m, 'test_tracked': test_m, 'seconds': epoch_times[-1]}
        history.append(rec)
        if d.main:
            if improved:
                save_atomic({'model': ema.module.state_dict(), 'args': vars(a), 'epoch': epoch + 1, 'val': val_m,
                             'split_hash': split_h}, out_dir / 'best.pth')
            save_atomic({'model': model.state_dict(), 'ema': ema.state_dict(), 'optimizer': optimizer.state_dict(),
                         'scaler': scaler.state_dict() if a.amp else None, 'epoch': epoch + 1, 'step': step,
                         'best_iou': best_iou, 'best_epoch': best_epoch, 'history': history,
                         'args': {**vars(a), 'steps_per_epoch': steps_per_epoch}, 'split_hash': split_h},
                        out_dir / 'last.pth')
            write_json(history, out_dir / 'history.json')
            msg = (f'[ep {epoch + 1:3d}/{a.epochs}] loss {tr["loss"]:.4f} (seg {tr["seg"]:.4f} bnd {tr["bnd"]:.4f} '
                   f'con {tr["con"]:.4f}' + (f' dis {tr["dis"]:.4f}' if model.distill_head is not None else '')
                   + f') lr {optimizer.param_groups[0]["lr"]:.2e}')
            if val_m:
                msg += (f' | val IoU {100 * val_m["iou"]:.2f} F1 {100 * val_m["f1"]:.2f} P {100 * val_m["precision"]:.1f} '
                        f'R {100 * val_m["recall"]:.1f} | best {100 * best_iou:.2f}@{best_epoch}')
            if test_m:
                msg += f' | test(track) IoU {100 * test_m["iou"]:.2f}'
            if n_skip:
                msg += f' | amp-skipped {n_skip}'
            if model.fm_fallbacks:
                msg += f' | FM FP32 fallbacks {model.fm_fallbacks}'
            mean_ep = float(np.mean(epoch_times[-3:]))
            msg += f' | {epoch_times[-1] / 60:.1f} min/ep, ETA {mean_ep * (a.epochs - epoch - 1) / 3600:.2f} h'
            print(msg, flush=True)
            if epoch == start_epoch:
                proj = (time.time() - t_start) + mean_ep * (a.epochs - epoch - 1)
                print(f'[train] projected total training time {proj / 3600:.2f} h (budget {a.max_train_hours} h)'
                      + ('  -> will stop early; resume in a new session with --resume' if proj > a.max_train_hours * 3600 else ''),
                      flush=True)
        d.barrier()
        if a.stop_after_epochs and epoch + 1 - start_epoch >= a.stop_after_epochs and epoch + 1 < a.epochs:
            log(d, f'[train] --stop-after-epochs {a.stop_after_epochs}: stopping as if out of time')
            stopped_early = True
            break

    # ---- choose the weights to evaluate ----
    schedule_complete = not stopped_early and (len(history) == 0 or history[-1]['epoch'] == a.epochs)
    status = {'tag': a.tag or out_dir.name, 'schedule_complete': schedule_complete,
              'epochs_done': history[-1]['epoch'] if history else 0, 'epochs': a.epochs, 'evaluated': False,
              'train_hours_total': sum(float(h.get('seconds') or 0.0) for h in history) / 3600}
    if not schedule_complete and a.no_eval_on_timeout:
        if d.main:
            write_json(status, out_dir / 'status.json')
        log(d, f'[train] stopped by the time budget after epoch {status["epochs_done"]}/{a.epochs}; final evaluation '
               f'skipped (--no-eval-on-timeout). Resume with --resume {out_dir / "last.pth"}')
        return
    if has_val and (out_dir / 'best.pth').exists():
        ck = torch.load(out_dir / 'best.pth', map_location='cpu', weights_only=True)
        ema.module.load_state_dict(ck['model'])
        weights = f'EMA weights of epoch {ck["epoch"]} (best validation IoU {100 * ck["val"]["iou"]:.2f})'
        del ck
    else:
        weights = f'final EMA weights (epoch {history[-1]["epoch"] if history else 0})'
        if d.main:
            save_atomic({'model': ema.module.state_dict(), 'args': vars(a),
                         'epoch': history[-1]['epoch'] if history else 0, 'val': None, 'split_hash': split_h},
                        out_dir / 'final.pth')
    d.barrier()
    log(d, f'[eval] evaluating {weights}')
    loaders = {'val': dl_val, 'test': make_eval_loader(rows['test'], a, d)}
    info = {'subset': subset, 'weights': weights, 'schedule_complete': schedule_complete,
            'select_threshold': has_val and not a.full_train,
            'train_hours': (time.time() - t_start) / 3600, 'train_hours_total': status['train_hours_total'],
            'epochs_done': status['epochs_done']}
    final_evaluation(a, d, ema.module, rows, loaders, out_dir, info)
    if d.main:
        write_json({**status, 'evaluated': True}, out_dir / 'status.json')
    if not schedule_complete:
        log(d, f'NOTE: the {a.epochs}-epoch schedule is incomplete; resume with --resume {out_dir / "last.pth"}')


def main(argv=None):
    a = parse_args(argv)
    d = Dist()
    try:
        set_seed(a.seed + 1000 * d.rank)
        if d.device.type == 'cuda':
            torch.backends.cudnn.benchmark = True
        out_dir = Path(a.out)
        if d.main:
            out_dir.mkdir(parents=True, exist_ok=True)
        d.barrier()
        rows, subset = load_rows(a)
        if a.eval_only:
            eval_only(a, d, rows, subset, out_dir)
        else:
            train(a, d, rows, subset, out_dir)
        d.barrier()
    finally:
        d.close()


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/ablation.py
#!/usr/bin/env python3
"""
ablation.py - experiment registry, Kaggle session queue and results collector for DPF-Net.

  python ablation.py list
  python ablation.py weights --queue A6 U6          # pre-download the encoder + foundation models of a queue
  python ablation.py run --work /kaggle/working --queue A6 A0 A5 [--prev /kaggle/input] [--session-hours 11.5]
  python ablation.py collect --roots /kaggle/working/runs /kaggle/input --out /kaggle/working/ablation

run      Runs the queued experiments one after another (torchrun on all GPUs). Runs found below --prev (outputs of
         earlier notebook versions attached as inputs) are carried forward into <work>/runs; an unfinished run is
         resumed from its last.pth. Each run gets the session time that is left minus a reserve for its final
         evaluation; when the session is nearly over the queue stops cleanly and the next session (with this
         output attached as input) continues it. --import ID=best.pth evaluates an existing checkpoint (e.g. the
         earlier SP-UNet runs) as experiment ID instead of training it again.
collect  Builds the ablation / comparison tables (Markdown, CSV, LaTeX), paired-bootstrap significance tests on
         per-tile counts, per-region tables and figures from every results.json found below --roots. Also works
         offline on downloaded run folders (only results.json, history.json and per_tile_test.csv are read).
"""
import argparse
import csv
import json
import math
import os
import shlex
import shutil
import subprocess
import sys
import time
from collections import OrderedDict
from pathlib import Path

HERE = Path(__file__).resolve().parent
FM_SAT = 'vit_large_patch16_dinov3.sat493m'   # DINOv3 ViT-L/16 distilled from the satellite (SAT-493M) 7B model
FM_WEB = 'vit_large_patch16_dinov3.lvd1689m'   # same architecture, web-image (LVD-1689M) pretraining
TEST_TILES = {'SAT': 2218, 'UAV': 4034}
VAL_TILES = {'SAT': 521, 'UAV': 941}
TORCHRUN_ENV = {'NCCL_P2P_DISABLE': '1', 'NCCL_IB_DISABLE': '1', 'OMP_NUM_THREADS': '2', 'PYTHONIOENCODING': 'utf-8'}
SKIP_DIRS = {'img', 'imgs', 'image', 'images', 'mask', 'masks', 'label', 'labels', 'vis_test', '__pycache__',
             '.git', 'splits'}

# ----------------------------------------------------------------------------------------
# experiment registry  (every row: ConvNeXt-T DINOv3 encoder, same schedule, seed 42, same evaluation)
# ----------------------------------------------------------------------------------------
EXPERIMENTS = OrderedDict()
BASE = ['--no-hires', '--no-prior', '--boundary-weight', '0', '--contrast-weight', '0']
FULL = ['--fm', FM_SAT, '--fm-mode', 'fuse', '--fm-fusion', 'gated', '--hard-mining', 'error+fm']


def _exp(eid, subset, title, args, group, priority):
    EXPERIMENTS[eid] = {'id': eid, 'subset': subset, 'title': title, 'args': list(args), 'group': group,
                        'priority': priority}


_exp('A0', 'SAT', 'UNet + PPM baseline', ['--fm', 'none'] + BASE, 'component', 1)
_exp('A1', 'SAT', '+ spectral-structural prior & high-res stem',
     ['--fm', 'none', '--boundary-weight', '0', '--contrast-weight', '0'], 'component', 2)
_exp('A2', 'SAT', '+ boundary head', ['--fm', 'none', '--contrast-weight', '0'], 'component', 2)
_exp('A3', 'SAT', '+ error-mined pixel contrast (= SP-UNet)', ['--fm', 'none'], 'component', 1)
_exp('A4', 'SAT', '+ foundation semantic prior, additive fusion',
     ['--fm', FM_SAT, '--fm-mode', 'fuse', '--fm-fusion', 'add', '--hard-mining', 'error'], 'component', 2)
_exp('A5', 'SAT', '+ structure-aware gated fusion',
     ['--fm', FM_SAT, '--fm-mode', 'fuse', '--fm-fusion', 'gated', '--hard-mining', 'error'], 'component', 1)
_exp('A6', 'SAT', '+ semantic hard-negative mining = DPF-Net', FULL + ['--ms-tta'], 'component', 1)
_exp('A7', 'SAT', 'DPF-Net without structural prior / high-res stem',
     FULL + ['--no-hires', '--no-prior'], 'complementarity', 1)
_exp('A8', 'SAT', 'DPF-Lite: foundation model only during training',
     ['--fm', FM_SAT, '--fm-mode', 'distill', '--hard-mining', 'error+fm', '--ms-tta'], 'efficiency', 2)
_exp('A9', 'SAT', 'DPF-Net with a web-image foundation model',
     ['--fm', FM_WEB, '--fm-mode', 'fuse', '--fm-fusion', 'gated', '--hard-mining', 'error+fm'], 'pretraining', 2)
_exp('S1', 'SAT', 'DPF-Net, seed 1', FULL + ['--ms-tta', '--seed', '1'], 'seeds', 3)
_exp('S2', 'SAT', 'DPF-Net, seed 2', FULL + ['--ms-tta', '--seed', '2'], 'seeds', 3)
_exp('U0', 'UAV', 'UAV: UNet + PPM baseline', ['--fm', 'none'] + BASE, 'uav', 2)
_exp('U3', 'UAV', 'UAV: SP-UNet', ['--fm', 'none'], 'uav', 1)
_exp('U6', 'UAV', 'UAV: DPF-Net', FULL + ['--ms-tta'], 'uav', 1)
_exp('U8', 'UAV', 'UAV: DPF-Lite', ['--fm', FM_SAT, '--fm-mode', 'distill', '--hard-mining', 'error+fm', '--ms-tta'],
     'uav', 3)

# comparisons tested with the paired bootstrap (reference, candidate): candidate - reference
PAIRS = [('A0', 'A1'), ('A1', 'A2'), ('A2', 'A3'), ('A3', 'A4'), ('A4', 'A5'), ('A5', 'A6'), ('A3', 'A5'),
         ('A3', 'A6'), ('A0', 'A6'), ('A7', 'A6'), ('A8', 'A6'), ('A3', 'A8'), ('A9', 'A6'),
         ('U0', 'U3'), ('U3', 'U6'), ('U0', 'U6'), ('U8', 'U6'), ('U3', 'U8')]
CHAIN = {'SAT': ['A0', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9'], 'UAV': ['U0', 'U3', 'U6', 'U8']}
LEGACY = {'fm': 'none', 'fm_mode': 'fuse', 'fm_fusion': 'gated', 'hard_mining': 'error'}


def fm_enabled(name):
    return name not in (None, '', 'none')


def garg(g, key):
    return g.get(key, LEGACY.get(key))


def short_fm(name):
    if name == FM_SAT:
        return 'DINOv3-SAT ViT-L'
    if name == FM_WEB:
        return 'DINOv3-LVD ViT-L'
    return name


def get_exp(eid):
    if eid not in EXPERIMENTS:
        raise SystemExit(f'unknown experiment "{eid}"; known: {", ".join(EXPERIMENTS)}')
    return EXPERIMENTS[eid]


def map_fm(args, fm_map):
    """Replaces foundation-model names (smoke tests use a tiny ViT instead of ViT-L)."""
    out = list(args)
    for i in range(len(out) - 1):
        if out[i] == '--fm' and out[i + 1] in fm_map:
            out[i + 1] = fm_map[out[i + 1]]
    return out


def parse_map(items, what):
    m = {}
    for it in items or []:
        if '=' not in it:
            raise SystemExit(f'{what}: expected KEY=VALUE, got "{it}"')
        k, v = it.split('=', 1)
        m[k.strip()] = v.strip()
    return m


def split_args(s):
    """Command-line words of a string; keeps Windows backslashes (POSIX rules elsewhere, e.g. Kaggle)."""
    return shlex.split(s or '', posix=os.name != 'nt')


def load_json(p):
    try:
        return json.loads(Path(p).read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return None


def write_json(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2), encoding='utf-8')
    os.replace(tmp, path)


def stream(cmd, env=None):
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(shlex.quote(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         encoding='utf-8', errors='replace', env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end='', flush=True)
    return p.wait()


# ----------------------------------------------------------------------------------------
# run folders
# ----------------------------------------------------------------------------------------
def find_run_dirs(roots, max_depth=5):
    """Folders containing results.json / last.pth / status.json below the roots (bounded, skips data folders)."""
    found, seen = [], set()
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        stack = [(root, 0)]
        while stack:
            p, depth = stack.pop()
            try:
                entries = list(os.scandir(p))
            except OSError:
                continue
            names = {e.name for e in entries}
            if names & {'results.json', 'last.pth', 'status.json'}:
                key = str(p.resolve())
                if key not in seen:
                    seen.add(key)
                    found.append(p)
                continue
            if depth >= max_depth:
                continue
            for e in entries:
                if e.is_dir(follow_symlinks=False) and e.name.lower() not in SKIP_DIRS:
                    stack.append((Path(e.path), depth + 1))
    return found


def run_tag(p):
    for f in ('results.json', 'status.json'):
        j = load_json(Path(p) / f)
        if j and j.get('tag'):
            return j['tag']
    return Path(p).name


def run_state(p):
    """('done' | 'partial' | 'evaluated-incomplete' | 'empty', epochs done)."""
    p = Path(p)
    if not p.is_dir():
        return 'empty', 0
    res = load_json(p / 'results.json')
    hist = load_json(p / 'history.json') or []
    ep = len(hist) if isinstance(hist, list) else 0
    if res is not None and res.get('schedule_complete') is not False:
        return 'done', ep
    if (p / 'last.pth').exists():
        return 'partial', ep
    if res is not None:
        return 'evaluated-incomplete', ep
    return 'empty', ep


def _rank(state):
    return {'done': 3, 'evaluated-incomplete': 1, 'partial': 2, 'empty': 0}[state[0]] * 10000 + state[1]


def carry_forward(prev_roots, runs_dir, keep_last=False):
    """Copies registry runs found in earlier outputs into runs_dir (the more advanced copy wins)."""
    runs_dir = Path(runs_dir).resolve()
    for p in find_run_dirs(prev_roots):
        rp = Path(p).resolve()
        if rp == runs_dir or runs_dir in rp.parents:
            continue
        tag = run_tag(p)
        if tag not in EXPERIMENTS:
            continue
        dst = runs_dir / tag
        st_src, st_dst = run_state(p), run_state(dst)
        if dst.exists() and _rank(st_dst) >= _rank(st_src):
            continue
        if dst.exists():
            shutil.rmtree(dst)
        skip = [] if (keep_last or st_src[0] != 'done') else ['last.pth']
        shutil.copytree(p, dst, ignore=shutil.ignore_patterns(*skip, '*.tmp'))
        print(f'[carry] {tag}: {st_src[0]} ({st_src[1]} epochs) from {p}', flush=True)


# ----------------------------------------------------------------------------------------
# list / weights
# ----------------------------------------------------------------------------------------
def cmd_list(a):
    print(f'{"id":4} {"subset":6} {"prio":4} {"group":15} title / extra arguments')
    for e in EXPERIMENTS.values():
        print(f'{e["id"]:4} {e["subset"]:6} {e["priority"]:<4} {e["group"]:15} {e["title"]}')
        print(f'{"":31} {" ".join(e["args"])}')


def resolved_args(eid, a, extra=()):
    sys.path.insert(0, str(HERE))
    import train_landslide as T  # noqa: E402  (heavy import only for run / weights)
    e = get_exp(eid)
    argv = ['--splits', 'x', '--out', 'y'] + map_fm(e['args'], parse_map(getattr(a, 'fm_map', None), '--fm-map'))
    return T.parse_args(argv + list(extra))


def cmd_weights(a):
    import timm
    extra = split_args(a.extra)
    names = []
    for eid in a.queue:
        ta = resolved_args(eid, a, extra)
        for n in [ta.encoder] + ([ta.fm] if fm_enabled(ta.fm) else []):
            if n not in names:
                names.append(n)
    for n in names:
        t0 = time.time()
        m = timm.create_model(n, pretrained=True)
        print(f'weights OK: {n} ({sum(p.numel() for p in m.parameters()) / 1e6:.1f} M params, '
              f'{time.time() - t0:.0f} s)', flush=True)
        del m


# ----------------------------------------------------------------------------------------
# run (queue)
# ----------------------------------------------------------------------------------------
def gpu_count():
    try:
        import torch
        return torch.cuda.device_count()
    except Exception:  # noqa: BLE001
        return 0


def launcher(nproc):
    if nproc > 1:
        return [sys.executable, '-m', 'torch.distributed.run', '--standalone', f'--nproc_per_node={nproc}']
    return [sys.executable]


def eval_reserve_hours(ta, subset, nproc):
    """Conservative estimate of the final evaluation time (T4, FP16): validation sweep + test, all TTA passes."""
    passes = 1 if ta.no_tta else (12 if ta.ms_tta else 8)
    fuse = fm_enabled(ta.fm) and ta.fm_mode == 'fuse'
    ips = (9.0 if fuse else 16.0) * max(1, nproc)  # single-pass images / s
    n = VAL_TILES.get(subset, 1000) + TEST_TILES.get(subset, 3000)
    return n * passes / ips / 3600 * 1.15 + 0.12


def ensure_splits(work, subset, prep_args):
    sp = Path(work) / 'splits' / subset
    if (sp / 'train.csv').exists() and (sp / 'test.csv').exists():
        return sp
    rc = stream([sys.executable, HERE / 'prepare_data.py', '--subset', subset, '--out', Path(work) / 'splits']
                + split_args(prep_args))
    if rc != 0:
        raise SystemExit(f'prepare_data.py failed for {subset} (exit {rc})')
    return sp


def cmd_run(a):
    t0 = a.t0 if a.t0 else time.time()
    deadline = t0 + a.session_hours * 3600
    work = Path(a.work)
    runs = work / 'runs'
    runs.mkdir(parents=True, exist_ok=True)
    prev = [p for p in (a.prev or []) if Path(p).is_dir()]
    if prev:
        carry_forward(prev, runs, a.keep_last)
    imports = parse_map(a.imports, '--import')
    fm_map = parse_map(a.fm_map, '--fm-map')
    extra = split_args(a.extra)
    nproc = a.nproc or max(1, gpu_count())
    queue = list(a.queue)
    if a.quick:
        queue = queue[:1]
    summary = []
    for eid in queue:
        e = get_exp(eid)
        ta = resolved_args(eid, a, extra)
        tag = eid + ('_quick' if a.quick else '')
        rd = runs / tag
        state, ep_done = run_state(rd)
        if state == 'done':
            summary.append((tag, 'done (already finished)'))
            continue
        sp = ensure_splits(work, e['subset'], a.prep_args)
        if state == 'empty' and eid in imports:  # evaluate an existing checkpoint as this experiment
            ck = Path(imports[eid])
            if not ck.is_file():
                summary.append((tag, f'IMPORT FAILED: {ck} not found'))
                continue
            left = (deadline - time.time()) / 3600
            need = eval_reserve_hours(ta, e['subset'], nproc)
            if left < need:
                summary.append((tag, f'import not started: {left:.2f} h left, ~{need:.2f} h needed'))
                break
            rd.mkdir(parents=True, exist_ok=True)
            cmd = (launcher(nproc) + [HERE / 'train_landslide.py', '--eval-only', '--eval-ckpts', ck, '--splits', sp,
                                      '--out', rd, '--tag', tag, '--batch-size', a.batch_size]
                   + (['--ms-tta'] if ta.ms_tta else []) + extra)
            rc = stream(cmd, TORCHRUN_ENV)
            if (ck.parent / 'history.json').exists() and not (rd / 'history.json').exists():
                shutil.copy2(ck.parent / 'history.json', rd / 'history.json')
            summary.append((tag, 'imported (evaluated)' if rc == 0 else f'IMPORT FAILED (exit {rc})'))
            continue
        left = (deadline - time.time()) / 3600
        reserve = a.eval_reserve_hours if a.eval_reserve_hours is not None else eval_reserve_hours(ta, e['subset'], nproc)
        budget = left - reserve
        if budget < a.min_start_hours:
            summary.append((tag, f'not started: {left:.2f} h left in this session (eval reserve {reserve:.2f} h)'))
            break
        cmd = (launcher(nproc) + [HERE / 'train_landslide.py', '--splits', sp, '--out', rd, '--tag', tag,
                                  '--epochs', 1 if a.quick else a.epochs, '--batch-size', a.batch_size,
                                  '--max-train-hours', f'{budget:.3f}', '--no-eval-on-timeout']
               + map_fm(e['args'], fm_map) + extra)
        if state == 'partial':
            cmd += ['--resume', rd / 'last.pth']
            print(f'[queue] {tag}: resuming after epoch {ep_done}', flush=True)
        print(f'[queue] {tag}: {e["title"]} | {left:.2f} h left, training budget {budget:.2f} h, '
              f'evaluation reserve {reserve:.2f} h', flush=True)
        t_run = time.time()
        rc = stream(cmd, TORCHRUN_ENV)
        st2, ep2 = run_state(rd)
        took = (time.time() - t_run) / 3600
        if rc != 0:
            summary.append((tag, f'FAILED (exit {rc}) after {took:.2f} h - see the log above'))
            if a.stop_on_error:
                break
            continue
        if st2 == 'done':
            summary.append((tag, f'done ({ep2} epochs, {took:.2f} h this session)'))
            if not a.keep_last and (rd / 'last.pth').exists() and not a.quick:
                (rd / 'last.pth').unlink()  # finished: keep best.pth / results only (saves output space)
            continue
        summary.append((tag, f'paused after epoch {ep2} (session budget) - re-run with this output attached'))
        break
    done_ids = {t for t, s in summary}
    for eid in queue:
        if eid not in done_ids and eid + '_quick' not in done_ids:
            summary.append((eid, 'waiting (next session)'))
    print('\n[queue] summary')
    for tag, s in summary:
        print(f'  {tag:8s} {s}')
    log = load_json(work / 'queue_log.json') or []
    log.append({'time': time.strftime('%Y-%m-%d %H:%M:%S'), 'queue': queue, 'summary': summary})
    write_json(log, work / 'queue_log.json')


# ----------------------------------------------------------------------------------------
# collect
# ----------------------------------------------------------------------------------------
def discover(roots):
    runs = {}
    for p in find_run_dirs(roots):
        res = load_json(Path(p) / 'results.json')
        if not res or 'main' not in res:
            continue
        tag = res.get('tag') or Path(p).name
        cand = {'dir': Path(p), 'res': res, 'mtime': (Path(p) / 'results.json').stat().st_mtime}
        old = runs.get(tag)
        key = (res.get('schedule_complete') is not False, cand['mtime'])
        if old is None or key > (old['res'].get('schedule_complete') is not False, old['mtime']):
            runs[tag] = cand
    for r in runs.values():
        h = load_json(r['dir'] / 'history.json')
        r['hist'] = h if isinstance(h, list) else []
    return runs


def margs(r):
    return r['res'].get('model_args') or r['res']['args']


def abl_key(res):
    return res.get('ablation_key') or ('test_TTA@t*' if res.get('tta') else 'test_noTTA@t*')


def components(r):
    g = margs(r)
    struct = 'yes' if not g.get('no_prior') else ('stem only' if not g.get('no_hires') else 'no')
    bnd = 'yes' if g.get('boundary_weight', 0) > 0 else 'no'
    con = garg(g, 'hard_mining') if g.get('contrast_weight', 0) > 0 else 'no'
    fm = garg(g, 'fm')
    if not fm_enabled(fm):
        sem = 'no'
    elif garg(g, 'fm_mode') == 'distill':
        sem = f'{short_fm(fm)}, training only'
    else:
        sem = f'{short_fm(fm)}, {garg(g, "fm_fusion")}'
    return {'struct': struct, 'boundary': bnd, 'contrast': con, 'semantic': sem}


def title_of(tag, r):
    return EXPERIMENTS[tag]['title'] if tag in EXPERIMENTS else r['res'].get('run', tag)


def read_tiles(r, mode):
    f = r['dir'] / (r['res'].get('per_tile_file') or 'per_tile_test.csv')
    if not f.exists():
        return None
    out = {}
    with open(f, newline='', encoding='utf-8') as fh:
        for row in csv.DictReader(fh):
            try:
                out[row['name']] = [float(row[f'{mode}_{k}']) for k in ('tp', 'fp', 'fn', 'tn')]
            except KeyError:
                return None
    return out


def _iou(s):
    import numpy as np
    den = s[..., 0] + s[..., 1] + s[..., 2]
    return np.where(den > 0, s[..., 0] / np.maximum(den, 1e-12), 0.0)


def bootstrap(A, B=None, n_boot=10000, seed=0, chunk=250):
    """Tile-level bootstrap of the dataset-level IoU (A, B: n x 4 count arrays of the same tiles, paired)."""
    import numpy as np
    rng = np.random.default_rng(seed)
    n = A.shape[0]
    ia, ib, done = [], [], 0
    while done < n_boot:
        k = min(chunk, n_boot - done)
        idx = rng.integers(0, n, size=(k, n))
        ia.append(_iou(A[idx].sum(axis=1)))
        if B is not None:
            ib.append(_iou(B[idx].sum(axis=1)))
        done += k
    ia = np.concatenate(ia)
    out = {'iou': float(_iou(A.sum(0))), 'ci': [float(v) for v in np.percentile(ia, [2.5, 97.5])]}
    if B is not None:
        ib = np.concatenate(ib)
        dd = ib - ia
        out.update({'iou_b': float(_iou(B.sum(0))), 'delta': float(_iou(B.sum(0)) - _iou(A.sum(0))),
                    'delta_ci': [float(v) for v in np.percentile(dd, [2.5, 97.5])],
                    'p': float(min(1.0, 2.0 * min((dd <= 0).mean(), (dd >= 0).mean())))})
    return out


def holm(ps):
    """Holm-Bonferroni adjusted p-values (same order as given)."""
    m = len(ps)
    order = sorted(range(m), key=lambda i: ps[i])
    adj, run = [0.0] * m, 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * ps[i]))
        adj[i] = run
    return adj


def pct(v, nd=2):
    return '-' if v is None or (isinstance(v, float) and math.isnan(v)) else f'{100 * v:.{nd}f}'


def fmt_p(p, n_boot):
    return f'<{1.0 / n_boot:.0e}' if p == 0 else (f'{p:.4f}' if p < 0.01 else f'{p:.3f}')


def tex_escape(s):
    return (str(s).replace('\\', '\\textbackslash{}').replace('&', '\\&').replace('%', '\\%')
            .replace('_', '\\_').replace('#', '\\#'))


def cmd_collect(a):
    import numpy as np
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    runs = discover(a.roots)
    if not runs:
        print('collect: no results.json found below', a.roots)
        return
    order = [t for t in EXPERIMENTS if t in runs] + sorted(t for t in runs if t not in EXPERIMENTS)
    summary = {'runs': {}, 'pairs': [], 'seeds': {}}

    # ---- per-run rows -------------------------------------------------------------------------------
    rows = []
    for tag in order:
        r = runs[tag]
        res = r['res']
        g = margs(r)
        m = res[abl_key(res)]
        main = res.get('main') or m
        st = res.get('model_stats') or {}
        subset = (res.get('subset') or EXPERIMENTS.get(tag, {}).get('subset') or 'SAT')
        base_tag = 'A0' if subset == 'SAT' else 'U0'
        base = runs.get(base_tag)
        d0 = (m['iou'] - base['res'][abl_key(base['res'])]['iou']) if base and base_tag != tag else None
        comp = components(r)
        row = OrderedDict([
            ('id', tag), ('subset', subset), ('configuration', title_of(tag, r)),
            ('struct_prior', comp['struct']), ('boundary', comp['boundary']), ('contrast_mining', comp['contrast']),
            ('semantic_prior', comp['semantic']), ('epochs', res.get('epochs_done') or g.get('epochs')),
            ('complete', res.get('schedule_complete')), ('eval_setting', abl_key(res).replace('test_', '')),
            ('P', m['precision']), ('R', m['recall']), ('F1', m['f1']), ('IoU', m['iou']), ('mIoU', m['miou']),
            ('dIoU_vs_baseline', d0), ('main_setting', res.get('main_key', '').replace('test_', '')),
            ('main_IoU', main['iou']), ('main_F1', main['f1']),
            ('params_M', st.get('params_M')), ('inference_params_M', st.get('inference_params_M')),
            ('gflops', st.get('gflops')), ('fps', st.get('fps_batch8')),
            ('train_hours', res.get('train_hours_total', res.get('train_hours')))])
        rows.append(row)
        summary['runs'][tag] = {k: v for k, v in row.items()}
        summary['runs'][tag]['dir'] = str(r['dir'])

    with open(out / 'ablation_table.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(list(rows[0].keys()))
        for row in rows:
            w.writerow(['' if v is None else (f'{v:.6f}' if isinstance(v, float) else v) for v in row.values()])

    def num(v, nd=1):
        return '-' if v is None else f'{v:.{nd}f}'

    md = ['# Ablation and comparison (test split)', '',
          'IoU / F1 / P / R in % for the landslide class from dataset-level pixel counts. "Eval" is the setting '
          'compared across rows (validation-selected threshold, D4-TTA unless stated); "Main" is the '
          'validation-selected main setting of each run (may add multi-scale TTA). dIoU = IoU - baseline IoU '
          '(A0 for SAT, U0 for UAV), percentage points.', '']
    for subset in ('SAT', 'UAV'):
        sub = [r for r in rows if r['subset'] == subset]
        if not sub:
            continue
        md += [f'## {subset}', '',
               '| ID | Configuration | Struct. prior | Boundary | Contrast mining | Semantic prior | Ep. | P | R | F1 | '
               'IoU | dIoU | Main IoU | Train. params (M) | Infer. params (M) | GFLOPs | FPS | Train h |',
               '|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|']
        for r in sub:
            d = '-' if r['dIoU_vs_baseline'] is None else f'{100 * r["dIoU_vs_baseline"]:+.2f}'
            inc = '' if r['complete'] is not False else ' (incomplete)'
            md.append(f'| {r["id"]} | {r["configuration"]}{inc} | {r["struct_prior"]} | {r["boundary"]} | '
                      f'{r["contrast_mining"]} | {r["semantic_prior"]} | {r["epochs"]} | {pct(r["P"])} | {pct(r["R"])} | '
                      f'{pct(r["F1"])} | **{pct(r["IoU"])}** | {d} | {pct(r["main_IoU"])} ({r["main_setting"]}) | '
                      f'{num(r["params_M"], 2)} | {num(r["inference_params_M"], 1)} | {num(r["gflops"])} | '
                      f'{num(r["fps"])} | {num(r["train_hours"], 2)} |')
        md.append('')

    # ---- LaTeX (booktabs) ---------------------------------------------------------------------------
    tex = ['% requires \\usepackage{booktabs,amssymb}', '\\begin{tabular}{llccccrrr}', '\\toprule',
           'ID & Configuration & Struct. & Bnd. & Contrast & Semantic & IoU (\\%) & F1 (\\%) & $\\Delta$IoU \\\\',
           '\\midrule']
    yes = {'yes': '\\checkmark', 'no': '--', 'stem only': 'stem'}
    for r in rows:
        d = '--' if r['dIoU_vs_baseline'] is None else f'{100 * r["dIoU_vs_baseline"]:+.2f}'
        tex.append(f'{tex_escape(r["id"])} & {tex_escape(r["configuration"])} & {yes.get(r["struct_prior"], "")} & '
                   f'{yes.get(r["boundary"], "")} & {tex_escape(r["contrast_mining"]) if r["contrast_mining"] != "no" else "--"} & '
                   f'{tex_escape(r["semantic_prior"]) if r["semantic_prior"] != "no" else "--"} & {pct(r["IoU"])} & '
                   f'{pct(r["F1"])} & {d} \\\\')
    tex += ['\\bottomrule', '\\end{tabular}']
    (out / 'ablation_table.tex').write_text('\n'.join(tex) + '\n', encoding='utf-8')

    # ---- paired bootstrap ---------------------------------------------------------------------------
    tiles = {}
    for tag in order:
        mode = abl_key(runs[tag]['res']).split('_', 1)[1].split('@')[0]
        tiles[tag] = read_tiles(runs[tag], mode)
    ci = {}
    for tag in order:
        if tiles[tag]:
            names = sorted(tiles[tag])
            A = np.array([tiles[tag][n] for n in names])
            ci[tag] = bootstrap(A, n_boot=a.n_boot, seed=1)['ci']
            summary['runs'][tag]['IoU_ci95'] = ci[tag]
    sig = ['## Paired bootstrap tests (resampling test tiles; dIoU = candidate - reference)', '',
           f'{a.n_boot} bootstrap samples of the test tiles; 95% percentile interval; two-sided bootstrap p-value and '
           'its Holm adjustment over all comparisons listed. Both runs evaluated in the "Eval" setting of the '
           'ablation table.', '',
           '| Reference | Candidate | IoU ref | IoU cand | dIoU (pp) | 95% CI (pp) | p | p (Holm) | tiles |',
           '|---|---|---|---|---|---|---|---|---|']
    for ref, cand in PAIRS:
        if not (tiles.get(ref) and tiles.get(cand)):
            continue
        names = sorted(set(tiles[ref]) & set(tiles[cand]))
        if len(names) < 10:
            continue
        A = np.array([tiles[ref][n] for n in names])
        B = np.array([tiles[cand][n] for n in names])
        b = bootstrap(A, B, n_boot=a.n_boot, seed=7)
        summary['pairs'].append({'reference': ref, 'candidate': cand, **b, 'tiles': len(names)})
    for pr_, ph in zip(summary['pairs'], holm([pr_['p'] for pr_ in summary['pairs']])):
        pr_['p_holm'] = ph
        sig.append(f'| {pr_["reference"]} | {pr_["candidate"]} | {pct(pr_["iou"])} | {pct(pr_["iou_b"])} | '
                   f'{100 * pr_["delta"]:+.2f} | [{100 * pr_["delta_ci"][0]:+.2f}, {100 * pr_["delta_ci"][1]:+.2f}] | '
                   f'{fmt_p(pr_["p"], a.n_boot)} | {fmt_p(ph, a.n_boot)} | {pr_["tiles"]} |')
    if not summary['pairs']:
        sig.append('| (no pair with per-tile counts for both runs yet) | | | | | | | | |')
    md += sig + ['']

    # ---- seeds ------------------------------------------------------------------------------------
    seed_tags = [t for t in ('A6', 'S1', 'S2') if t in runs]
    if len(seed_tags) >= 2:
        for key, label in (('IoU', 'Eval'), ('main_IoU', 'Main')):
            vals = [summary['runs'][t][key] for t in seed_tags]
            mu = sum(vals) / len(vals)
            sd = math.sqrt(sum((v - mu) ** 2 for v in vals) / (len(vals) - 1))
            summary['seeds'][key] = {'runs': seed_tags, 'mean': mu, 'std': sd}
            md.append(f'Seeds ({", ".join(seed_tags)}), {label} IoU: {100 * mu:.2f} +- {100 * sd:.2f} (mean +- s.d.)')
        md.append('')

    # ---- per-region ---------------------------------------------------------------------------------
    for subset in ('SAT', 'UAV'):
        tags = [t for t in order if summary['runs'][t]['subset'] == subset]
        if not tags:
            continue
        per = {}
        for t in tags:
            res = runs[t]['res']
            mode = abl_key(res).split('_', 1)[1].split('@')[0]
            per[t] = (res.get('per_region_by_mode') or {}).get(mode) or res.get('per_region') or {}
        regions = sorted({rg for t in tags for rg in per[t]})
        if not regions:
            continue
        hdr = '| Region | tiles | ' + ' | '.join(tags) + ' |'
        md += [f'## Per-region IoU, {subset} (Eval setting)', '', hdr, '|' + '---|' * (len(tags) + 2)]
        prc = []
        for rg in regions:
            n_t = next((per[t][rg].get('tiles') for t in tags if rg in per[t]), '')
            vals = [pct(per[t][rg]['iou']) if rg in per[t] else '-' for t in tags]
            md.append(f'| {rg} | {n_t} | ' + ' | '.join(vals) + ' |')
            prc.append([rg, n_t] + [per[t][rg]['iou'] if rg in per[t] else '' for t in tags])
        md.append('')
        with open(out / f'per_region_{subset}.csv', 'w', newline='', encoding='utf-8') as f:
            w = csv.writer(f)
            w.writerow(['region', 'tiles'] + tags)
            w.writerows(prc)
        summary[f'per_region_{subset}'] = {t: {rg: per[t][rg]['iou'] for rg in per[t]} for t in tags}

    (out / 'ablation.md').write_text('\n'.join(md) + '\n', encoding='utf-8')
    write_json(summary, out / 'summary.json')
    print('\n'.join(md))
    try:
        make_figures(runs, summary, ci, out)
    except Exception as e:  # noqa: BLE001  (figures are optional; tables are the record)
        print(f'collect: figures skipped ({type(e).__name__}: {e})')
    print(f'\ncollect: wrote {out}/ablation.md, ablation_table.csv/.tex, summary.json, per_region_*.csv, fig_*.png/.pdf')


# ----------------------------------------------------------------------------------------
# figures (static, light surface; validated categorical palette, fixed slot order)
# ----------------------------------------------------------------------------------------
SLOTS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']        # blue, orange, aqua, yellow (validated, light mode)
INK, INK2, MUTED, GRID, AXIS, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#ffffff'
MARKERS = ['o', 's', 'D', '^']


def _style(ax):
    ax.set_facecolor(SURFACE)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color(AXIS)
        ax.spines[s].set_linewidth(0.8)
    ax.tick_params(colors=INK2, labelsize=8, length=0)
    ax.grid(True, color=GRID, linewidth=0.6, linestyle='-')
    ax.set_axisbelow(True)


def _save(fig, out, name):
    for ext in ('png', 'pdf'):
        fig.savefig(out / f'{name}.{ext}', dpi=300, bbox_inches='tight', facecolor=SURFACE)


def make_figures(runs, summary, ci, out):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.rcParams.update({'font.family': 'sans-serif', 'font.size': 8, 'axes.labelcolor': INK2,
                         'text.color': INK, 'axes.titlesize': 9})
    for subset, keys in (('SAT', ['A6', 'A3', 'A0', 'A8']), ('UAV', ['U6', 'U3', 'U0', 'U8'])):
        series = [k for k in keys if k in runs]
        names = {k: (EXPERIMENTS[k]['title'].split('= ')[-1] if k in EXPERIMENTS else k) for k in series}
        names.update({'A6': 'DPF-Net', 'A3': 'SP-UNet', 'A0': 'Baseline', 'A8': 'DPF-Lite',
                      'U6': 'DPF-Net', 'U3': 'SP-UNet', 'U0': 'Baseline', 'U8': 'DPF-Lite'})
        # 1. validation IoU per epoch (line chart, <= 4 series, legend)
        curves = [(k, [(h['epoch'], h['val']['iou']) for h in runs[k]['hist'] if h.get('val')]) for k in series]
        curves = [(k, c) for k, c in curves if len(c) >= 2]
        if curves:
            fig, ax = plt.subplots(figsize=(5.0, 3.0))
            _style(ax)
            for i, (k, c) in enumerate(curves):
                slot = series.index(k)
                ax.plot([e for e, _ in c], [100 * v for _, v in c], color=SLOTS[slot], linewidth=1.5,
                        solid_capstyle='round', label=f'{names[k]} ({k})')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Validation IoU (%), no TTA, t = 0.5')
            ax.set_title(f'CAS-{subset}: validation IoU during training', loc='left', color=INK)
            lo = min(100 * v for _, c in curves for _, v in c[len(c) // 4:])
            ax.set_ylim(max(0.0, lo - 3), None)  # zoom on the converged part; the early ramp is cut
            longest = max((c for _, c in curves), key=len)
            ax.set_xlim(longest[len(longest) // 4][0], longest[-1][0] + 0.5)
            ax.legend(frameon=False, fontsize=8, loc='lower right')
            _save(fig, out, f'fig_val_curves_{subset}')
            plt.close(fig)
        # 2. per-region dot plot (<= 3 series: all-pairs palette cap; marker shape as second channel)
        per = summary.get(f'per_region_{subset}') or {}
        dots = [k for k in series[:3] if k in per]
        if dots:
            regions = sorted({rg for k in dots for rg in per[k]}, key=lambda rg: per[dots[0]].get(rg, 0.0))
            fig, ax = plt.subplots(figsize=(5.2, 0.32 * len(regions) + 0.9))
            _style(ax)
            ax.grid(True, axis='x', color=GRID, linewidth=0.6)
            ax.grid(False, axis='y')
            for y, rg in enumerate(regions):
                vals = [100 * per[k][rg] for k in dots if rg in per[k]]
                if len(vals) > 1:
                    ax.plot([min(vals), max(vals)], [y, y], color=AXIS, linewidth=1.2, zorder=1)
            for i, k in enumerate(dots):
                ys = [y for y, rg in enumerate(regions) if rg in per[k]]
                xs = [100 * per[k][rg] for rg in regions if rg in per[k]]
                ax.scatter(xs, ys, s=30, color=SLOTS[series.index(k)], marker=MARKERS[i], edgecolors=SURFACE,
                           linewidths=1.0, zorder=3, label=f'{names[k]} ({k})')
            ax.set_yticks(range(len(regions)))
            ax.set_yticklabels(regions, color=INK2)
            ax.set_xlabel('Test IoU (%)')
            ax.set_title(f'CAS-{subset}: per-region test IoU', loc='left', color=INK)
            ax.legend(frameon=False, fontsize=8, loc='lower right')
            _save(fig, out, f'fig_per_region_{subset}')
            plt.close(fig)
    # 3. ablation forest plot (single series: one hue, no legend; whiskers = 95% bootstrap CI)
    for subset in ('SAT', 'UAV'):
        chain = [t for t in CHAIN[subset] if t in runs and t in ci]
        if len(chain) < 2:
            continue
        fig, ax = plt.subplots(figsize=(5.2, 0.34 * len(chain) + 0.9))
        _style(ax)
        ax.grid(True, axis='x', color=GRID, linewidth=0.6)
        ax.grid(False, axis='y')
        ys = list(range(len(chain)))[::-1]
        for y, t in zip(ys, chain):
            v = 100 * summary['runs'][t]['IoU']
            lo, hi = 100 * ci[t][0], 100 * ci[t][1]
            ax.plot([lo, hi], [y, y], color=SLOTS[0], linewidth=1.5, solid_capstyle='round', zorder=2)
            ax.scatter([v], [y], s=34, color=SLOTS[0], edgecolors=SURFACE, linewidths=1.0, zorder=3)
            if t in ('A0', 'A6', 'U0', 'U6'):
                ax.annotate(f'{v:.2f}', (hi, y), xytext=(5, 0), textcoords='offset points', va='center',
                            fontsize=8, color=INK2)
        ax.set_yticks(ys)
        ax.set_yticklabels([f'{t}  {EXPERIMENTS[t]["title"] if t in EXPERIMENTS else t}' for t in chain], color=INK2)
        ax.set_xlabel('Test IoU (%) with 95% bootstrap interval')
        ax.set_title(f'CAS-{subset}: ablation', loc='left', color=INK)
        _save(fig, out, f'fig_ablation_{subset}')
        plt.close(fig)


# ----------------------------------------------------------------------------------------
def build_parser():
    p = argparse.ArgumentParser(description='DPF-Net experiment queue and results collector')
    sub = p.add_subparsers(dest='cmd', required=True)
    sub.add_parser('list', help='print the experiment registry')
    w = sub.add_parser('weights', help='download the encoder + foundation-model weights a queue needs')
    w.add_argument('--queue', nargs='+', required=True)
    w.add_argument('--extra', default='')
    w.add_argument('--fm-map', nargs='*', default=[])
    r = sub.add_parser('run', help='run a queue of experiments within the session budget')
    r.add_argument('--queue', nargs='+', required=True, help='experiment ids, in order (see "list")')
    r.add_argument('--work', default='/kaggle/working')
    r.add_argument('--prev', nargs='*', default=['/kaggle/input'], help='folders with earlier outputs (carried forward)')
    r.add_argument('--epochs', type=int, default=80)
    r.add_argument('--batch-size', type=int, default=8, help='per GPU')
    r.add_argument('--session-hours', type=float, default=11.5, help='hard session limit minus a safety margin')
    r.add_argument('--t0', type=float, default=0.0, help='session start (time.time()); default: now')
    r.add_argument('--min-start-hours', type=float, default=0.4, help='do not start / resume a run with less time')
    r.add_argument('--eval-reserve-hours', type=float, default=None, help='override the evaluation-time estimate')
    r.add_argument('--nproc', type=int, default=0, help='processes (default: number of GPUs)')
    r.add_argument('--extra', default='', help='extra train_landslide.py arguments for every run (use --extra=...)')
    r.add_argument('--import', dest='imports', nargs='*', default=[], metavar='ID=CKPT',
                   help='evaluate an existing checkpoint as experiment ID instead of training it')
    r.add_argument('--prep-args', default='', help='extra prepare_data.py arguments (use --prep-args=...)')
    r.add_argument('--fm-map', nargs='*', default=[], metavar='OLD=NEW', help='replace foundation models (tests)')
    r.add_argument('--keep-last', action='store_true', help='keep last.pth of finished runs')
    r.add_argument('--stop-on-error', action='store_true')
    r.add_argument('--quick', action='store_true', help='1 epoch of the first queued experiment (<id>_quick)')
    c = sub.add_parser('collect', help='build ablation tables, significance tests and figures')
    c.add_argument('--roots', nargs='+', required=True)
    c.add_argument('--out', required=True)
    c.add_argument('--n-boot', type=int, default=10000)
    return p


def main(argv=None):
    a = build_parser().parse_args(argv)
    {'list': cmd_list, 'weights': cmd_weights, 'run': cmd_run, 'collect': cmd_collect}[a.cmd](a)


if __name__ == '__main__':
    main()

In [ ]:
import os, sys, json, glob, shlex, subprocess
import torch

def run(args, env=None):
    """Run a command (argument list, no shell), stream its output into the notebook, raise on failure."""
    args = [str(x) for x in args]
    print('$', ' '.join(shlex.quote(x) for x in args), flush=True)
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         encoding='utf-8', errors='replace', env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f'command failed with exit code {rc}')

print('python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA', torch.version.cuda)
N_GPU = torch.cuda.device_count()
print('GPUs:', N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
if N_GPU == 0:
    raise SystemExit('No GPU visible: Settings -> Accelerator -> GPU T4 x2')

# timm >= 1.0.20 is needed for the DINOv3 models; checked in a fresh interpreter so an upgrade takes effect
CHECK = ("import sys, timm; have = set(timm.list_pretrained('*dinov3*')); "
         "need = ['convnext_tiny.dinov3_lvd1689m', 'vit_large_patch16_dinov3.sat493m']; "
         "print('timm', timm.__version__); sys.exit(0 if all(n in have for n in need) else 3)")
if subprocess.run([sys.executable, '-c', CHECK]).returncode != 0:
    run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'timm>=1.0.20'])
    if subprocess.run([sys.executable, '-c', CHECK]).returncode != 0:
        raise SystemExit('timm does not provide the DINOv3 models even after upgrading (Internet must be On).')

# download the encoder and the foundation-model weights once (ViT-L: 1.2 GB), so every process finds them cached
run([sys.executable, f'{WORK}/ablation.py', 'weights', '--queue', *QUEUE, f'--extra={EXTRA_ARGS}'])
run([sys.executable, f'{WORK}/ablation.py', 'list'])

In [ ]:
# Train + evaluate the queued experiments (DDP on all GPUs). Safe to re-run: finished runs are skipped.
cmd = [sys.executable, f'{WORK}/ablation.py', 'run', '--work', WORK, '--queue', *QUEUE, '--epochs', EPOCHS,
       '--batch-size', BATCH_PER_GPU, '--session-hours', SESSION_HOURS, '--t0', NB_T0, f'--extra={EXTRA_ARGS}',
       '--prev', *PREV]
if IMPORT:
    cmd += ['--import'] + [f'{k}={v}' for k, v in IMPORT.items()]
if QUICK_TEST:
    cmd.append('--quick')
t0 = time.time()
run(cmd)
print(f'queue finished in {(time.time() - t0) / 3600:.2f} h')

In [ ]:
# Ablation / comparison tables, paired-bootstrap significance tests, per-region tables and figures
run([sys.executable, f'{WORK}/ablation.py', 'collect', '--roots', f'{WORK}/runs', *PREV, '--out', f'{WORK}/ablation'])
from IPython.display import Markdown, Image, display
for tag in QUEUE:
    f = f'{WORK}/runs/{tag}/results.md'
    if os.path.exists(f):
        display(Markdown(open(f, encoding='utf-8').read()))
for f in sorted(glob.glob(f'{WORK}/ablation/fig_*.png')):
    display(Image(filename=f))
print('download for the paper:', sorted(os.listdir(f'{WORK}/ablation')))

## After the run

* **Continue the queue**: open a new version, *Add Input → Your Work →* this notebook's latest output, keep or extend
  `QUEUE`, *Save & Run All*. Finished runs are not repeated.
* **What to keep for the paper**: `/kaggle/working/ablation/` holds `ablation.md`, `ablation_table.csv` / `.tex`,
  `summary.json`, `per_region_*.csv` and `fig_*.png` / `.pdf`. Each run keeps `results.md` / `results.json`,
  `history.json`, `per_tile_test.csv`, `best.pth` and `vis_test/`.
* **Two accounts**: run different queues, download each `runs/<ID>` folder (the small files are enough), put them in
  one folder and run locally: `python ablation.py collect --roots <folder> --out ablation`.
* **Single custom run**: `python train_landslide.py --help` lists every option. `--fm none` gives SP-UNet,
  `--fm-mode distill` gives DPF-Lite, and `--ms-tta` adds multi-scale TTA (chosen on validation).